> **VERSIÓN CON SOLUCIONES — solo para el equipo docente.** No distribuir a los estudiantes.

# Tutorial 14 — De reconocer a imaginar: modelos generativos y multimodales
**CC5209 · Visión Computacional** · Duración: **180 min** (con una pausa sugerida después de la sección 3)

Hasta ahora, todos los modelos del curso miraban una imagen y respondían algo sobre ella: una clase, una caja, una máscara. Hoy damos vuelta la flecha. Primero vamos a construir **máquinas que inventan imágenes**, y después vamos a **conectar imágenes y lenguaje** para buscar, describir y segmentar con palabras.

**Hilo conductor.**
- **Parte A — imaginar (secciones 1–4).** Con el mismo dataset (Fashion-MNIST) entrenamos cuatro máquinas de inventar prendas: un autoencoder, un VAE, una GAN y un modelo de difusión. Las medimos con la misma regla y vemos por qué la difusión ganó. Después abrimos **Stable Diffusion** y descubrimos que son las mismas piezas, escaladas.
- **Parte B — conectar con el lenguaje (secciones 5–7).** **CLIP** clasifica sin etiquetas y busca imágenes con frases; **BLIP-2** y **LLaVA** conversan sobre una foto; **SAM + CLIP** segmenta lo que nombras.
- **Cierre.** Un circuito completo: texto → imagen → texto → máscara.

| # | Sección | Min | Ejercicios |
|---|---|---|---|
| 0 | Preparación, datos y el "juez" | 5 | — |
| 1 | Autoencoder y VAE: comprimir no es imaginar | 25 | **E1** `reparametrizar` · **E2** `perdida_vae` |
| 2 | GAN: un juego entre dos redes | 15 | **E3** `perdidas_gan` |
| 3 | Difusión desde cero: ruido, pasos y guía | 35 | **E4** `difundir` · **E5** `paso_ddpm` · **E6** `guiar` · **E7** `distancia_frechet` |
| — | ☕ *Pausa* | 10 | |
| 4 | Stable Diffusion por dentro | 25 | **E8** `generar_a_mano` |
| 5 | CLIP: imágenes y frases en el mismo espacio | 30 | **E9** `perdida_clip` · **E10** `zero_shot` |
| 6 | Conversar con imágenes: BLIP-2 y LLaVA | 15 | **E11** `proyectar` |
| 7 | SAM + CLIP: segmentar lo que nombras | 15 | **E12** `recortar_mascara` |
| 8 | Cierre: el circuito completo | 5 | — |

**Cómo funciona este notebook** (igual que los anteriores):
- Cada ejercicio trae **solo pistas**. Escribe tu código donde dice `# TU CÓDIGO AQUÍ` y borra el `return None`.
- Debajo de cada ejercicio hay una celda que lo **verifica**: ✅ correcto, ❌ algo falla, ⬜ todavía no lo implementas.
- Si un ejercicio no está listo, el notebook usa una **implementación de respaldo** (⚠️) o se **salta** la celda (⏭). Así puedes correrlo completo y volver después.
- Pensado para **Colab con GPU (T4)**: *Entorno de ejecución → Cambiar tipo de entorno → T4 GPU*. En CPU también corre, con una configuración reducida (menos datos, menos épocas, Stable Diffusion a 256 px): los resultados se ven peores y algunas celdas tardan varios minutos.

In [ ]:
# 0.1 Instalación, importaciones y configuración (≈1 min)
import importlib.util, subprocess, sys
for paquete in ['diffusers', 'ultralytics']:
    if importlib.util.find_spec(paquete) is None:
        print(f'Instalando {paquete}...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', paquete], check=True)

import os, io, gc, math, time, copy, random, zipfile, urllib.request, warnings
from pathlib import Path
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager
import torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image, ImageDraw, ImageFont
from scipy import stats
warnings.filterwarnings('ignore')
import transformers, diffusers
transformers.logging.set_verbosity_error(); diffusers.logging.set_verbosity_error()
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
transformers.utils.logging.disable_progress_bar(); diffusers.utils.logging.disable_progress_bar()

GPU = torch.cuda.is_available()
dispositivo = torch.device('cuda' if GPU else 'cpu')
if not GPU:
    torch.set_flush_denormal(True)   # evita que la CPU se arrastre con números diminutos
torch.manual_seed(0); np.random.seed(0); random.seed(0)

CFG = dict(
    n_train=60000 if GPU else 8000,      # Fashion-MNIST para entrenar
    n_test=10000 if GPU else 2000,
    ep_juez=3 if GPU else 2,
    ep_ae=10 if GPU else 4,
    ep_gan=15 if GPU else 5,
    ep_dif=15 if GPU else 6,
    n_eval=2000 if GPU else 300,         # muestras por modelo en la comparación final
    lote_gen=1000 if GPU else 300,
    n_guia=50 if GPU else 12,            # muestras por clase en el experimento de guía
    sd_lado=512 if GPU else 256,
    sd_pasos=25 if GPU else 8,
    sd_pasos_prueba=10 if GPU else 4,
    clip_k=100 if GPU else 20,           # imágenes de prueba por clase en zero-shot
    sam='sam_b.pt' if GPU else 'mobile_sam.pt',
    sam_grilla=16 if GPU else 8,
)
print(f'Dispositivo: {dispositivo}' + ('' if GPU else '  ⚠️ sin GPU: se usa la configuración reducida'))
print('torch', torch.__version__, '| transformers', transformers.__version__, '| diffusers', diffusers.__version__)

In [ ]:
# 0.2 Datos: Fashion-MNIST, CIFAR-10, Imagenette y COCO128 (≈300 MB, todo desde GitHub)
DATOS = Path('datos'); DATOS.mkdir(exist_ok=True)
URL = 'https://github.com/ultralytics/assets/releases/download/v0.0.0/'

def bajar(nombre):
    destino = DATOS / nombre
    for intento in range(3):
        if destino.exists():
            break
        try:
            urllib.request.urlretrieve(URL + nombre, destino)
        except Exception as e:
            print(f'Reintentando {nombre}: {e}'); time.sleep(3)
    return destino

ZIPS = {n: zipfile.ZipFile(bajar(n + '.zip')) for n in ['fashion-mnist', 'cifar10', 'imagenette160', 'coco128']}

def leer_carpeta(z, raiz, por_clase, semilla=0, modo='RGB'):
    """Lee `por_clase` imágenes de cada subcarpeta raiz/<clase>/ del zip. Devuelve (imágenes PIL, etiquetas, clases)."""
    archivos = {}
    for n in z.namelist():
        p = n.split('/')
        if len(p) == 4 and n.startswith(raiz + '/') and p[3].lower().endswith(('.png', '.jpg', '.jpeg')):
            archivos.setdefault(p[2], []).append(n)
    clases = sorted(archivos)
    rng = random.Random(semilla)
    imagenes, etiquetas = [], []
    for i, c in enumerate(clases):
        lista = sorted(archivos[c]); rng.shuffle(lista)
        for n in lista[:por_clase]:
            imagenes.append(Image.open(io.BytesIO(z.read(n))).convert(modo)); etiquetas.append(i)
    return imagenes, np.array(etiquetas), clases

PRENDAS = ['camiseta', 'pantalón', 'suéter', 'vestido', 'abrigo', 'sandalia', 'camisa', 'zapatilla', 'bolso', 'botín']
PRENDAS_EN = ['t-shirt', 'trousers', 'pullover', 'dress', 'coat', 'sandal', 'shirt', 'sneaker', 'bag', 'ankle boot']

t0 = time.time()
_im, _y, _ = leer_carpeta(ZIPS['fashion-mnist'], 'fashion-mnist/train', CFG['n_train'] // 10, modo='L')
X_tr = torch.tensor(np.stack([np.array(i) for i in _im])).float().div(255).unsqueeze(1)   # (N,1,28,28) en [0,1]
Y_tr = torch.tensor(_y)
_im, _y, _ = leer_carpeta(ZIPS['fashion-mnist'], 'fashion-mnist/test', CFG['n_test'] // 10, modo='L')
X_te = torch.tensor(np.stack([np.array(i) for i in _im])).float().div(255).unsqueeze(1)
Y_te = torch.tensor(_y)
del _im, _y
_g = torch.Generator().manual_seed(0)                       # el zip viene ordenado por clase: barajamos una vez
_p = torch.randperm(len(X_tr), generator=_g); X_tr, Y_tr = X_tr[_p], Y_tr[_p]
_p = torch.randperm(len(X_te), generator=_g); X_te, Y_te = X_te[_p], Y_te[_p]
X_tr_d, Y_tr_d = X_tr.to(dispositivo), Y_tr.to(dispositivo)   # todo el entrenamiento cabe en la GPU
print(f'Fashion-MNIST: {len(X_tr)} de entrenamiento y {len(X_te)} de prueba ({time.time() - t0:.0f} s)')

def foto(nombre):
    """Una foto de COCO128 como imagen PIL."""
    return Image.open(io.BytesIO(ZIPS['coco128'].read(f'coco128/images/train2017/{nombre}.jpg'))).convert('RGB')
FOTO_GRUPO, FOTO_GATO, FOTO_JIRAFA = '000000000542', '000000000650', '000000000025'

In [ ]:
# 0.3 Herramientas: figuras y verificación
def grilla(x, filas=None, borde=1):
    """x: tensor (N,C,H,W) en [0,1] -> arreglo listo para imshow."""
    x = x.detach().float().cpu().clamp(0, 1)
    n, c, h, w = x.shape
    filas = filas or int(math.ceil(math.sqrt(n)))
    cols = int(math.ceil(n / filas))
    lienzo = torch.ones(c, filas * (h + borde) + borde, cols * (w + borde) + borde)
    for i in range(n):
        f, k = divmod(i, cols)
        lienzo[:, borde + f * (h + borde): borde + f * (h + borde) + h, borde + k * (w + borde): borde + k * (w + borde) + w] = x[i]
    a = lienzo.permute(1, 2, 0).numpy()
    return a[..., 0] if c == 1 else a

def mostrar(imagenes, titulos=None, tam=3.0, sup=None):
    """Muestra una lista de imágenes (PIL, arreglos o grillas) en una fila."""
    fig, ejes = plt.subplots(1, len(imagenes), figsize=(tam * len(imagenes), tam), squeeze=False)
    for i, (ax, im) in enumerate(zip(ejes[0], imagenes)):
        im = np.asarray(im)
        ax.imshow(im, cmap='gray' if im.ndim == 2 else None, vmin=0 if im.ndim == 2 else None, vmax=(1 if im.dtype != np.uint8 else 255) if im.ndim == 2 else None)
        ax.axis('off')
        if titulos: ax.set_title(titulos[i], fontsize=10)
    if sup: fig.suptitle(sup)
    plt.tight_layout(); plt.show()

ESTADO, _AVISADOS = {}, set()

def revisar(ej, base, pruebas):
    """base: función sin argumentos que llama a tu código. pruebas: lista de (descripción, función(resultado) -> bool)."""
    try:
        r = base()
    except NotImplementedError:
        r = None
    except Exception as e:
        ESTADO[ej] = '❌'
        print(f'❌ {ej}: tu código lanzó {type(e).__name__}: {e}')
        return False
    if r is None:
        ESTADO[ej] = '⬜'
        for d, _ in pruebas:
            print(f'⬜ {d}  (todavía no lo implementas)')
        return False
    ok = True
    for d, p in pruebas:
        try:
            b = bool(p(r))
        except Exception as e:
            b, d = False, f'{d}  [{type(e).__name__}: {e}]'
        print(('✅ ' if b else '❌ ') + d)
        ok = ok and b
    ESTADO[ej] = '✅' if ok else '❌'
    return ok

def funciona(ej):
    return ESTADO.get(ej) == '✅'

def usar(ej, nombre):
    """Tu función si pasó su verificación; si no, una implementación de referencia (con un aviso, una sola vez)."""
    if funciona(ej):
        return globals()[nombre]
    if ej not in _AVISADOS:
        print(f'⚠️ respaldo: {ej} no está verificado; uso una implementación de referencia de `{nombre}`.')
        _AVISADOS.add(ej)
    return _REF[nombre]

def pendiente(ej, que=''):
    print(f'⏭ Esta celda necesita el ejercicio {ej} resuelto' + (f' ({que}).' if que else '.'))

def memoria_gpu(etiqueta=''):
    """Libera la memoria que PyTorch tiene reservada y no usa, e informa cuánta queda ocupada."""
    gc.collect()
    if GPU:
        torch.cuda.empty_cache()
        print(f'GPU {etiqueta}: {torch.cuda.memory_allocated() / 1e9:.1f} GB ocupados de '
              f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
#@title 🔒 Referencias ocultas para los respaldos y las verificaciones (no hace falta abrir esta celda)
import base64 as _b64, zlib as _zl
_REF = {}
exec(_zl.decompress(_b64.b64decode('eNqVVl2L4zYUfc+vEPRh5ZnEtcO2hYFAWeiwfS0L+xCCUWw5EWNJriTP2PPQ396jDydOmMI2DIwi3XvuuedeXeUn8qfsOy65cqwWWnFLGk4Mb7nhqhaMUMtJrWVvBGwI/jpGat41jOh66Bwj1V9/PMOnI0o7ftT6JVsJ2WvjiGTuPK/VIPuJMEtUP285berzzZdcqbwdVO1AhHXe+nnVGi2JrQW8k2kncHharVYNb8G0Z4ZJ7ox4Z4bKYU06fXplJntaEXwMd4NRRA7kMcXgY0+L/BfyMBtiFU8MU42qOvHCgZOlAD03jWhY9co4hYNwdk3GNbkGWpMjd2xX5kUK+YXsyJjbM+v5vjgkFjU2n/MjqJupqo22toLkRvdT9SbcuYrQywiGN0NQYvfJDvJTRn4mXwLYSwesTUwhEsc5LZFgJEQ2Pt2HB7LFKm6FpLMrRFLF03oM9AH10vmY9RqL29RtdWIqMqsMZ11IHOuWdVannBtQoj+aX0KJ3H3LRc2vZ1kWQMPn8Ydli4Rm3HcO6yVwpJugT/+jHLe493wT7FLWZk3m9mwE+rkRho4FACDwIBpAsa5l1ZEZw5J+7AhC1929O+Svgr/RTbkmD3RfHlAgYOSNkDRDWcvsNiQ75vZv46hv5rGAaOiHDXaz63YIPVeWWV01TS/pWIEV721g5zvBRnZ2STLxTlwlbzAXUG/4IkhwAmE0Vwy6zGIRH0F8AwZwn188CIiiJY7sdqR4utR9vrc+1up+IzXtAuYuv9MgMAwQsRKq1qoJKVZpZWvWsdvxcLUEdDTwgs9OyOpqkV1Kax3zI7JqcW3O3NEWxWq3CbktIZHqc2a9FFM4bNzU8x02204z9+vnmH27vbPc/pelBIb01m2ZS45LWWQ+4rwONhY2NiHW+jXENfoNQ2D3jFbl8JhPtncnUREm3uEdh2xQV1Jbkt8BeimWsEKF5Dn15uidAW9ItqheBPH/riMwKB3yoZRK3ypym4U5lYUJlkF7UHOGAdf6eWb9BNv62gIoT5Mhqu+vd2XP2lEuj5WQJxQWCze6ucK7siguM1mE+660kawT73x2ivJSaIKLtduUMUP3gTGAPzZOedFL1who5fJvyEm3TrKRztbzg1XjEWOmkgwehlEh2YkrFDZ+9wuDjV2Ro3JM2A71+WaGWd7Jvw421ldp5XWgyTUSmjBpJrj6iTP6JpxsLoXyvP0KhLzQ3uCyPy72Y6NNoDHCVyhHjcYIo5GVvxWTL91UYAKtPzwf/flYzBPq65p8B1LiGN/Fj4l6Dn6v8K0xAd3T++qNQCxuRIsxWIzJ4ntwf/Qbi8c2yrqfiqcY42ksD2j7frrOnCjucubU+38SzztHZv2FpPh502UHoJfb3+4e0tW/Inzh+Q==')).decode(), _REF)
print('Referencias cargadas.')

In [ ]:
# 0.4 El "juez": un clasificador de Fashion-MNIST que usaremos como regla para medir lo que generen los modelos
class Juez(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(), nn.Linear(64 * 7 * 7, 128), nn.ReLU())
        self.cabeza = nn.Linear(128, 10)
    def forward(self, x):
        return self.cabeza(self.features(x))

def entrenar(modelo, paso, epocas, lote=128, lr=1e-3, nombre=''):
    """Bucle genérico sobre X_tr. `paso(modelo, xb, yb)` devuelve la pérdida del lote."""
    opt = torch.optim.Adam(modelo.parameters(), lr=lr)
    historia, t0 = [], time.time()
    for ep in range(epocas):
        perm = torch.randperm(len(X_tr_d), device=dispositivo)
        suma, n = 0.0, 0
        for i in range(0, len(perm), lote):
            idx = perm[i:i + lote]
            perdida = paso(modelo, X_tr_d[idx], Y_tr_d[idx])
            opt.zero_grad(); perdida.backward(); opt.step()
            suma += perdida.item() * len(idx); n += len(idx)
        historia.append(suma / n)
        print(f'{nombre} época {ep + 1}/{epocas}: pérdida {historia[-1]:.3f}  ({time.time() - t0:.0f} s)')
    return historia

torch.manual_seed(0)
juez = Juez().to(dispositivo)
entrenar(juez, lambda m, x, y: F.cross_entropy(m(x), y), CFG['ep_juez'], nombre='juez')
juez.eval()

@torch.no_grad()
def opinar(x01, lote=1000):
    """x01: (N,1,28,28) en [0,1]. Devuelve probabilidades (N,10) y features de la penúltima capa (N,128), en CPU."""
    P, Fs = [], []
    for i in range(0, len(x01), lote):
        f = juez.features(x01[i:i + lote].to(dispositivo).float())
        Fs.append(f.cpu()); P.append(juez.cabeza(f).softmax(1).cpu())
    return torch.cat(P), torch.cat(Fs)

def resumen_juez(x01):
    """confianza: probabilidad media de la clase ganadora (¿parecen prendas?).
    cobertura: entropía del histograma de clases / log(10) (1 = las 10 prendas aparecen por igual)."""
    P, _ = opinar(x01)
    hist = torch.bincount(P.argmax(1), minlength=10).float() / len(P)
    entropia = -(hist[hist > 0] * hist[hist > 0].log()).sum().item()
    return dict(confianza=P.max(1).values.mean().item(), cobertura=entropia / math.log(10))

P_te, _ = opinar(X_te)
print(f'Exactitud del juez en prueba: {(P_te.argmax(1) == Y_te).float().mean():.1%}')
print('Juez sobre prendas reales:', {k: round(v, 3) for k, v in resumen_juez(X_te).items()})

ejemplos = torch.cat([X_te[Y_te == c][:8] for c in range(10)])
plt.figure(figsize=(7, 9)); plt.imshow(grilla(ejemplos, filas=10), cmap='gray'); plt.axis('off')
for c in range(10):
    plt.text(-3, 14.5 + c * 29, PRENDAS[c], ha='right', va='center', fontsize=10)
plt.title('Fashion-MNIST: 10 prendas, 28×28 en escala de grises'); plt.show()

## 1 · Autoencoder y VAE: comprimir no es lo mismo que imaginar (25 min)

Un **autoencoder** tiene dos partes: un *codificador* que comprime la imagen a un código corto $z$, y un *decodificador* que intenta reconstruirla a partir de $z$. Se entrena para que la salida se parezca a la entrada.

Para poder **ver** el espacio de códigos, vamos a usar $z$ de solo **2 dimensiones**: cada prenda queda convertida en un punto del plano. La pregunta del día es simple: si tomo un punto cualquiera de ese plano y lo decodifico, **¿obtengo una prenda nueva?**

In [ ]:
# 1.1 Un autoencoder con código de 2 dimensiones
class Codificador(nn.Module):
    def __init__(self, salida):
        super().__init__()
        self.red = nn.Sequential(nn.Flatten(), nn.Linear(784, 512), nn.ReLU(), nn.Linear(512, 256), nn.ReLU(), nn.Linear(256, salida))
    def forward(self, x):
        return self.red(x)

class Decodificador(nn.Module):
    def __init__(self, d=2):
        super().__init__()
        self.red = nn.Sequential(nn.Linear(d, 256), nn.ReLU(), nn.Linear(256, 512), nn.ReLU(), nn.Linear(512, 784))
    def forward(self, z):
        return self.red(z).view(-1, 1, 28, 28)        # logits: sigmoid(logits) es la imagen

class Autoencoder(nn.Module):
    def __init__(self, d=2):
        super().__init__()
        self.enc, self.dec = Codificador(d), Decodificador(d)
    def forward(self, x):
        z = self.enc(x)
        return self.dec(z), z

torch.manual_seed(0)
ae = Autoencoder().to(dispositivo)
# Pérdida de reconstrucción: entropía cruzada binaria píxel a píxel, sumada sobre la imagen y promediada en el lote
entrenar(ae, lambda m, x, y: F.binary_cross_entropy_with_logits(m(x)[0], x, reduction='sum') / len(x), CFG['ep_ae'], nombre='autoencoder')
ae.eval();

In [ ]:
# 1.2 El espacio de códigos del autoencoder y lo que pasa al muestrear "al azar"
with torch.no_grad():
    Z_ae = ae.enc(X_te.to(dispositivo)).cpu()
    Z_ae_tr = ae.enc(X_tr_d[:5000]).cpu()
media_ae, cov_ae = Z_ae_tr.mean(0), torch.cov(Z_ae_tr.T)   # la mejor gaussiana que calza con los códigos

lo, hi = np.percentile(Z_ae.numpy(), 1, axis=0), np.percentile(Z_ae.numpy(), 99, axis=0)
g1, g2 = np.linspace(lo[0], hi[0], 15), np.linspace(lo[1], hi[1], 15)
Z_mapa_ae = torch.tensor([[a, b] for b in g2[::-1] for a in g1], dtype=torch.float32)
Z_azar_ae = torch.distributions.MultivariateNormal(media_ae, cov_ae).sample((64,))
with torch.no_grad():
    mapa_ae = torch.sigmoid(ae.dec(Z_mapa_ae.to(dispositivo))).cpu()
    azar_ae = torch.sigmoid(ae.dec(Z_azar_ae.to(dispositivo))).cpu()

fig, ax = plt.subplots(1, 3, figsize=(17, 5.5))
sc = ax[0].scatter(Z_ae[:, 0], Z_ae[:, 1], c=Y_te, cmap='tab10', s=3)
ax[0].legend(sc.legend_elements()[0], PRENDAS, fontsize=7, markerscale=0.8, loc='best')
ax[0].scatter(Z_azar_ae[:, 0], Z_azar_ae[:, 1], c='k', marker='x', s=25)
ax[0].set_title('Códigos de prueba (×: 64 puntos al azar)')
ax[1].imshow(grilla(mapa_ae, filas=15), cmap='gray'); ax[1].axis('off'); ax[1].set_title('Decodificar una grilla regular del plano')
ax[2].imshow(grilla(azar_ae, filas=8), cmap='gray'); ax[2].axis('off'); ax[2].set_title('Decodificar los 64 puntos al azar')
plt.tight_layout(); plt.show()
print('Juez sobre 2.000 muestras al azar del autoencoder:', {k: round(v, 3) for k, v in resumen_juez(
    torch.sigmoid(ae.dec(torch.distributions.MultivariateNormal(media_ae, cov_ae).sample((2000,)).to(dispositivo))).detach()).items()})

Mira el panel izquierdo: las clases forman "brazos" y regiones de forma caprichosa, con **huecos** entre ellas. El decodificador solo aprendió a reconstruir los puntos donde cayeron imágenes de entrenamiento; en los huecos no tiene idea de qué dibujar, y por eso muchas muestras al azar son manchas o mezclas raras. **El autoencoder comprime, pero no está hecho para imaginar.**

### Variational Autoencoder (VAE)
El VAE cambia dos cosas:
1. El codificador ya no entrega un punto, sino una **distribución**: una media $\mu$ y una varianza $\sigma^2$ (en la práctica entrega $\log\sigma^2$, que puede ser cualquier número real). El decodificador recibe una **muestra** $z \sim \mathcal{N}(\mu, \sigma^2)$.
2. A la pérdida de reconstrucción se le suma una **divergencia KL** que empuja cada $\mathcal{N}(\mu, \sigma^2)$ hacia $\mathcal{N}(0, I)$. Así los códigos llenan el espacio de forma ordenada, sin huecos, y para generar basta con muestrear $z \sim \mathcal{N}(0, I)$.

Para poder entrenar con descenso de gradiente, la muestra se escribe como $z = \mu + \sigma \cdot \varepsilon$ con $\varepsilon \sim \mathcal{N}(0, I)$. Este es el **truco de reparametrización**: el azar queda en $\varepsilon$, y los gradientes llegan sin problema a $\mu$ y a $\sigma$.

In [ ]:
# 🧩 E1 · Truco de reparametrización
def reparametrizar(mu, logvar):
    """mu, logvar: tensores (B, d). Devuelve z = mu + sigma * eps, con eps ~ N(0, I) y la misma forma que mu."""
    return mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)

In [ ]:
# ✔️ Verificación E1
def _grad_llega():
    mu = torch.zeros(4, 2, requires_grad=True); lv = torch.zeros(4, 2, requires_grad=True)
    reparametrizar(mu, lv).sum().backward()
    return mu.grad is not None and torch.allclose(mu.grad, torch.ones(4, 2)) and lv.grad is not None and lv.grad.abs().sum() > 0

_mu = torch.tensor([[1.0, -2.0]]).repeat(20000, 1); _lv = torch.tensor([[0.0, math.log(4.0)]]).repeat(20000, 1)
revisar('E1', lambda: reparametrizar(_mu, _lv), [
    ('devuelve la misma forma que mu', lambda r: r.shape == _mu.shape),
    ('la media de las muestras es mu ≈ (1, -2)', lambda r: torch.allclose(r.mean(0), torch.tensor([1.0, -2.0]), atol=0.05)),
    ('la desviación de las muestras es sigma ≈ (1, 2)', lambda r: torch.allclose(r.std(0), torch.tensor([1.0, 2.0]), atol=0.05)),
    ('con varianza casi nula devuelve mu', lambda r: torch.allclose(reparametrizar(_mu[:5], torch.full((5, 2), -40.0)), _mu[:5], atol=1e-6)),
    ('los gradientes llegan a mu y a logvar', lambda r: _grad_llega()),
])

La pérdida del VAE tiene dos términos (promediados en el lote):

$$\mathcal{L} = \underbrace{\text{BCE}(\hat x, x)}_{\text{reconstrucción, sumada sobre los 784 píxeles}} \;+\; \beta\,\underbrace{\left(-\tfrac12 \sum_{j=1}^{d} \big(1 + \log\sigma_j^2 - \mu_j^2 - \sigma_j^2\big)\right)}_{\text{KL}\big(\mathcal{N}(\mu,\sigma^2)\,\|\,\mathcal{N}(0,I)\big)}$$

La KL vale 0 justo cuando $\mu = 0$ y $\sigma = 1$, y crece cuando el código se aleja del origen o se vuelve demasiado "seguro" ($\sigma \to 0$).

In [ ]:
# 🧩 E2 · Pérdida del VAE
def perdida_vae(logits, x, mu, logvar, beta=1.0):
    B = x.shape[0]
    rec = F.binary_cross_entropy_with_logits(logits, x, reduction='sum') / B
    kl = -0.5 * torch.sum(1 + logvar - mu ** 2 - logvar.exp()) / B
    return rec + beta * kl, rec, kl

In [ ]:
# ✔️ Verificación E2
torch.manual_seed(1)
_lg, _x = torch.randn(6, 1, 28, 28), torch.rand(6, 1, 28, 28)
_m, _v = torch.randn(6, 2), torch.randn(6, 2)
revisar('E2', lambda: perdida_vae(_lg, _x, _m, _v), [
    ('devuelve tres escalares (total, reconstruccion, kl)', lambda r: len(r) == 3 and all(torch.as_tensor(v).numel() == 1 for v in r)),
    ('la reconstrucción es la BCE sumada por imagen y promediada en el lote', lambda r: torch.allclose(r[1], _REF['perdida_vae'](_lg, _x, _m, _v)[1], rtol=1e-4)),
    ('KL = 0 cuando mu = 0 y sigma = 1', lambda r: abs(float(perdida_vae(_lg, _x, torch.zeros(6, 2), torch.zeros(6, 2))[2])) < 1e-5),
    ('KL = 1 cuando mu = (1, 1) y sigma = 1', lambda r: abs(float(perdida_vae(_lg, _x, torch.ones(6, 2), torch.zeros(6, 2))[2]) - 1.0) < 1e-5),
    ('la KL coincide con la referencia en un caso cualquiera', lambda r: torch.allclose(r[2], _REF['perdida_vae'](_lg, _x, _m, _v)[2], rtol=1e-4)),
    ('total = reconstrucción + beta·KL (con beta = 4)', lambda r: torch.allclose(perdida_vae(_lg, _x, _m, _v, beta=4.0)[0], r[1] + 4 * r[2], rtol=1e-5)),
])

In [ ]:
# 1.3 Entrenar el VAE (misma arquitectura que el autoencoder, pero el codificador entrega mu y logvar)
class VAE(nn.Module):
    def __init__(self, d=2):
        super().__init__()
        self.enc, self.dec = Codificador(2 * d), Decodificador(d)
    def forward(self, x):
        mu, logvar = self.enc(x).chunk(2, dim=1)
        z = usar('E1', 'reparametrizar')(mu, logvar)
        return self.dec(z), mu, logvar

def paso_vae(m, x, y):
    logits, mu, logvar = m(x)
    return usar('E2', 'perdida_vae')(logits, x, mu, logvar)[0]

torch.manual_seed(0)
vae = VAE().to(dispositivo)
entrenar(vae, paso_vae, CFG['ep_ae'], nombre='VAE')
vae.eval();

In [ ]:
# 1.4 El espacio del VAE: ordenado alrededor del origen, y "habitable"
with torch.no_grad():
    Z_vae = vae.enc(X_te.to(dispositivo)).chunk(2, dim=1)[0].cpu()     # usamos la media mu de cada prenda
    q = stats.norm.ppf(np.linspace(0.04, 0.96, 15))                      # cuantiles de N(0,1): cubren el espacio por igual
    Z_mapa_vae = torch.tensor([[a, b] for b in q[::-1] for a in q], dtype=torch.float32)
    mapa_vae = torch.sigmoid(vae.dec(Z_mapa_vae.to(dispositivo))).cpu()
    azar_vae = torch.sigmoid(vae.dec(torch.randn(64, 2, device=dispositivo))).cpu()

fig, ax = plt.subplots(1, 3, figsize=(17, 5.5))
sc = ax[0].scatter(Z_vae[:, 0], Z_vae[:, 1], c=Y_te, cmap='tab10', s=3)
ax[0].legend(sc.legend_elements()[0], PRENDAS, fontsize=7, markerscale=0.8, loc='best')
ax[0].add_patch(plt.Circle((0, 0), 2, fill=False, ls='--', color='k')); ax[0].set_aspect('equal')
ax[0].set_title('Medias mu de prueba (círculo: radio 2)')
ax[1].imshow(grilla(mapa_vae, filas=15), cmap='gray'); ax[1].axis('off'); ax[1].set_title('"Mapa de prendas": grilla de cuantiles de N(0, I)')
ax[2].imshow(grilla(azar_vae, filas=8), cmap='gray'); ax[2].axis('off'); ax[2].set_title('64 muestras z ~ N(0, I)')
plt.tight_layout(); plt.show()
with torch.no_grad():
    print('Juez sobre 2.000 muestras al azar del VAE:', {k: round(v, 3) for k, v in resumen_juez(torch.sigmoid(vae.dec(torch.randn(2000, 2, device=dispositivo)))).items()})

In [ ]:
# 1.5 Interpolar entre dos prendas: de un botín a una zapatilla
def tipica(c):
    """La prenda de la clase c que el juez reconoce con más seguridad (una "típica", no una rareza)."""
    idx = torch.nonzero(Y_te == c)[:, 0]
    return X_te[idx[P_te[idx, c].argmax()]][None].to(dispositivo)
a, b = tipica(9), tipica(7)
pesos = torch.linspace(0, 1, 10, device=dispositivo)[:, None]
with torch.no_grad():
    za, zb = ae.enc(a), ae.enc(b)
    fila_ae = torch.sigmoid(ae.dec((1 - pesos) * za + pesos * zb))
    ma, mb = vae.enc(a).chunk(2, 1)[0], vae.enc(b).chunk(2, 1)[0]
    fila_vae = torch.sigmoid(vae.dec((1 - pesos) * ma + pesos * mb))
plt.figure(figsize=(12, 3))
plt.imshow(grilla(torch.cat([fila_ae, fila_vae]), filas=2), cmap='gray'); plt.axis('off')
plt.title('Interpolación lineal en el espacio de códigos — arriba: autoencoder · abajo: VAE'); plt.show()

**Para conversar.** Compara las dos grillas de 64 muestras al azar y las dos interpolaciones. En el VAE, todo el "mapa" está ocupado por prendas que cambian de forma gradual, y la interpolación pasa por prendas intermedias en vez de atravesar zonas vacías. Pero sus muestras son **borrosas**: casi todas tienen forma de prenda, y aun así quedan lejos de la nitidez de una real (lo vas a medir en la sección 3.6). Dos razones: con solo 2 dimensiones caben pocos detalles, y la pérdida de reconstrucción premia "promediar" cuando el modelo duda entre varias opciones. El VAE es **estable y ordenado, pero difuso**. Guarda esta idea: Stable Diffusion usa un VAE (con un código mucho más grande) como primera pieza.

## 2 · GAN: un juego entre dos redes (15 min)

Una **GAN** no intenta reconstruir nada. Pone a competir dos redes:
- el **generador** $G$ transforma ruido $z$ en una imagen;
- el **discriminador** $D$ recibe una imagen y entrega un *logit*: alto si cree que es real y bajo si cree que es falsa.

$$\min_G \max_D \;\; \mathbb{E}_{x}\big[\log D(x)\big] + \mathbb{E}_{z}\big[\log\big(1 - D(G(z))\big)\big]$$

En la práctica, $D$ se entrena como un clasificador binario (reales = 1, falsas = 0). Para $G$ se usa el **truco no saturante**: en lugar de minimizar $\log(1 - D(G(z)))$, se minimiza $-\log D(G(z))$, es decir, se entrena a $G$ para que $D$ diga "real" sobre sus imágenes. La figura de abajo muestra por qué.

In [ ]:
# 2.1 ¿Por qué el truco no saturante? Gradiente que recibe G según cuánto lo "pilla" D
p = np.linspace(0.001, 0.999, 400)            # D(G(z)): probabilidad de "real" que D le da a una falsa
logit = np.log(p / (1 - p))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(p, np.log(1 - p), label='original: minimizar log(1 − D(G(z)))')
ax[0].plot(p, -np.log(p), label='no saturante: minimizar −log D(G(z))')
ax[0].set_xlabel('D(G(z))'); ax[0].set_title('Pérdida del generador'); ax[0].legend(fontsize=8)
ax[1].plot(p, p, label='original: |d/dlogit| = D(G(z))')          # d/dl log(1-σ(l)) = -σ(l)
ax[1].plot(p, 1 - p, label='no saturante: |d/dlogit| = 1 − D(G(z))')   # d/dl -log σ(l) = -(1-σ(l))
ax[1].axvspan(0, 0.1, color='r', alpha=0.1); ax[1].text(0.012, 0.5, 'inicio del\nentrenamiento:\nD gana fácil', fontsize=8)
ax[1].set_xlabel('D(G(z))'); ax[1].set_title('Tamaño del gradiente respecto al logit'); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

Al principio, $G$ produce basura y $D$ la reconoce enseguida: $D(G(z)) \approx 0$. Con la pérdida original el gradiente es casi **cero** justo cuando $G$ más necesita aprender. Con la versión no saturante, el gradiente es máximo en esa zona.

In [ ]:
# 🧩 E3 · Pérdidas de la GAN
def perdidas_gan(logit_real, logit_falso):
    d = (F.binary_cross_entropy_with_logits(logit_real, torch.ones_like(logit_real))
         + F.binary_cross_entropy_with_logits(logit_falso, torch.zeros_like(logit_falso)))
    g = F.binary_cross_entropy_with_logits(logit_falso, torch.ones_like(logit_falso))
    return d, g

In [ ]:
# ✔️ Verificación E3
def _grad_g_sube():
    lf = torch.zeros(5, requires_grad=True)
    perdidas_gan(torch.zeros(5), lf)[1].backward()
    return bool((lf.grad < 0).all())       # bajar la pérdida de G = subir el logit de sus falsas

_r, _f = torch.randn(7), torch.randn(7)
revisar('E3', lambda: perdidas_gan(_r, _f), [
    ('devuelve dos escalares', lambda r: len(r) == 2 and all(torch.as_tensor(v).numel() == 1 for v in r)),
    ('si D no sabe nada (logits 0): pérdida_D = 2·log 2 y pérdida_G = log 2',
     lambda r: [round(float(v), 4) for v in perdidas_gan(torch.zeros(4), torch.zeros(4))] == [round(2 * math.log(2), 4), round(math.log(2), 4)]),
    ('si D acierta con seguridad, pérdida_D ≈ 0 y pérdida_G ≈ 10',
     lambda r: float(perdidas_gan(torch.full((4,), 10.), torch.full((4,), -10.))[0]) < 1e-3 and abs(float(perdidas_gan(torch.full((4,), 10.), torch.full((4,), -10.))[1]) - 10) < 1e-3),
    ('el gradiente de G empuja hacia arriba el logit de las falsas', lambda r: _grad_g_sube()),
    ('coincide con la referencia en un caso cualquiera', lambda r: all(torch.allclose(a, b) for a, b in zip(r, _REF['perdidas_gan'](_r, _f)))),
])

In [ ]:
# 2.2 Una DCGAN pequeña. Trabaja con imágenes en [-1, 1] (por eso la Tanh al final del generador)
Z_GAN = 64
class Generador(nn.Module):
    def __init__(self, dz=Z_GAN):
        super().__init__()
        self.red = nn.Sequential(
            nn.Linear(dz, 128 * 7 * 7), nn.Unflatten(1, (128, 7, 7)), nn.BatchNorm2d(128), nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(),     # 7 -> 14
            nn.ConvTranspose2d(64, 1, 4, 2, 1), nn.Tanh())                           # 14 -> 28
    def forward(self, z):
        return self.red(z)

class Discriminador(nn.Module):
    def __init__(self):
        super().__init__()
        self.red = nn.Sequential(
            nn.Conv2d(1, 64, 4, 2, 1), nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2),
            nn.Flatten(), nn.Linear(128 * 7 * 7, 1))
    def forward(self, x):
        return self.red(x).squeeze(1)

torch.manual_seed(0)
G, D = Generador().to(dispositivo), Discriminador().to(dispositivo)
opt_g = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_d = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
z_fijo = torch.randn(32, Z_GAN, device=dispositivo)
perd = usar('E3', 'perdidas_gan')
hist_gan, fotos_gan, t0 = {'D': [], 'G': []}, {}, time.time()
for ep in range(CFG['ep_gan']):
    perm = torch.randperm(len(X_tr_d), device=dispositivo)
    sd_, sg_, n = 0.0, 0.0, 0
    for i in range(0, len(perm), 128):
        real = X_tr_d[perm[i:i + 128]] * 2 - 1
        falso = G(torch.randn(len(real), Z_GAN, device=dispositivo))
        ld, _ = perd(D(real), D(falso.detach()))                       # paso de D: las falsas no llevan gradiente a G
        opt_d.zero_grad(); ld.backward(); opt_d.step()
        _, lg = perd(torch.zeros(1, device=dispositivo), D(falso))     # paso de G: solo importa la parte de las falsas
        opt_g.zero_grad(); lg.backward(); opt_g.step()
        sd_ += ld.item() * len(real); sg_ += lg.item() * len(real); n += len(real)
    hist_gan['D'].append(sd_ / n); hist_gan['G'].append(sg_ / n)
    G.eval()
    with torch.no_grad():
        fotos_gan[ep + 1] = (G(z_fijo).cpu() + 1) / 2
    G.train()
    print(f'GAN época {ep + 1}/{CFG["ep_gan"]}: pérdida D {hist_gan["D"][-1]:.3f} · pérdida G {hist_gan["G"][-1]:.3f}  ({time.time() - t0:.0f} s)')
G.eval();

In [ ]:
# 2.3 Cómo evolucionan las muestras (mismo z en todas las épocas) y las pérdidas
eps_ver = sorted({1, max(1, CFG['ep_gan'] // 3), max(1, 2 * CFG['ep_gan'] // 3), CFG['ep_gan']})
fig, ax = plt.subplots(1, len(eps_ver) + 1, figsize=(4 * (len(eps_ver) + 1), 4))
for a, e in zip(ax, eps_ver):
    a.imshow(grilla(fotos_gan[e][:16], filas=4), cmap='gray'); a.axis('off'); a.set_title(f'época {e}')
ax[-1].plot(range(1, CFG['ep_gan'] + 1), hist_gan['D'], 'o-', label='D'); ax[-1].plot(range(1, CFG['ep_gan'] + 1), hist_gan['G'], 'o-', label='G')
ax[-1].set_xlabel('época'); ax[-1].legend(); ax[-1].set_title('Pérdidas: no "bajan", se disputan')
plt.tight_layout(); plt.show()
with torch.no_grad():
    print('Juez sobre 2.000 muestras de la GAN:', {k: round(v, 3) for k, v in resumen_juez((G(torch.randn(2000, Z_GAN, device=dispositivo)) + 1) / 2).items()})

**Para conversar.** Las prendas de la GAN son **nítidas**: nadie le pidió promediar, solo engañar a $D$. Pero fíjate en dos cosas:
- Las pérdidas no bajan de forma monótona como en todo lo que habíamos entrenado. No hay un número que diga "voy bien": es un **juego**, y puede oscilar o romperse.
- Mira la **cobertura**. Si es baja, el generador encontró unas pocas prendas que engañan bien a $D$ y se quedó ahí: es el **colapso de modo** (*mode collapse*). Con esta DCGAN y Fashion-MNIST no suele pasar, pero basta cambiar la tasa de aprendizaje o la arquitectura para provocarlo; ningún número de la pérdida te avisaría. En la sección 3.6 lo comparamos con los otros modelos.

## 3 · Difusión desde cero: ruido, pasos y guía (35 min)

La idea de los modelos de difusión es sorprendentemente simple:
- **Hacia adelante** (no se aprende): a una imagen le agregamos ruido gaussiano de a poco, en $T = 1000$ pasos, hasta que no queda nada de ella.
- **Hacia atrás** (se aprende): entrenamos una red que, dado $x_t$ y el paso $t$, **adivina el ruido** que se le agregó. Para generar partimos de ruido puro y lo vamos quitando paso a paso.

Con $\beta_t$ = cuánto ruido se agrega en el paso $t$, $\alpha_t = 1 - \beta_t$ y $\bar\alpha_t = \prod_{s \le t} \alpha_s$, el proceso hacia adelante tiene una forma cerrada: se puede saltar directo a cualquier paso,

$$x_t = \sqrt{\bar\alpha_t}\; x_0 + \sqrt{1 - \bar\alpha_t}\;\varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, I).$$

$\sqrt{\bar\alpha_t}$ es cuánto queda de la imagen y $\sqrt{1-\bar\alpha_t}$ es cuánto ruido hay. Trabajamos con imágenes en $[-1, 1]$.

In [ ]:
# 3.1 El horario de ruido (lineal, como en el paper de DDPM)
T = 1000
betas = torch.linspace(1e-4, 0.02, T, device=dispositivo)
alfas = 1 - betas
alfa_barra = torch.cumprod(alfas, dim=0)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].plot(alfa_barra.sqrt().cpu(), label='√ᾱ_t  (cuánto queda de la imagen)')
ax[0].plot((1 - alfa_barra).sqrt().cpu(), label='√(1−ᾱ_t)  (cuánto ruido hay)')
ax[0].set_xlabel('paso t'); ax[0].legend(fontsize=8)
ax[1].semilogy((alfa_barra / (1 - alfa_barra)).cpu()); ax[1].axhline(1, color='k', lw=0.6)
ax[1].set_xlabel('paso t'); ax[1].set_title('Señal / ruido (escala log)')
plt.tight_layout(); plt.show()
print(f'En t = 999 queda {alfa_barra[-1].sqrt():.4f} de la imagen original.')

In [ ]:
# 🧩 E4 · El proceso hacia adelante
def difundir(x0, t, ruido, alfa_barra):
    ab = alfa_barra[t].view(-1, 1, 1, 1)
    return ab.sqrt() * x0 + (1 - ab).sqrt() * ruido

In [ ]:
# ✔️ Verificación E4
torch.manual_seed(2)
_x0 = (X_tr[:8] * 2 - 1).to(dispositivo)
_ru = torch.randn_like(_x0)
_t = torch.tensor([0, 0, 999, 999, 500, 500, 250, 250], device=dispositivo)
def _var_en_500():
    x0, e = torch.randn(2000, 1, 28, 28, device=dispositivo), torch.randn(2000, 1, 28, 28, device=dispositivo)
    return abs(difundir(x0, torch.full((2000,), 500, device=dispositivo), e, alfa_barra).var().item() - 1) < 0.02
revisar('E4', lambda: difundir(_x0, _t, _ru, alfa_barra), [
    ('devuelve la forma de x0', lambda r: r.shape == _x0.shape),
    ('en t = 0 la imagen casi no cambia', lambda r: (r[:2] - _x0[:2]).abs().mean() < 0.02),
    ('en t = 999 casi solo queda ruido (correlación > 0,99)', lambda r: torch.corrcoef(torch.stack([r[2:4].flatten(), _ru[2:4].flatten()]))[0, 1] > 0.99),
    ('cada imagen usa su propio paso t', lambda r: torch.allclose(r, _REF['difundir'](_x0, _t, _ru, alfa_barra), atol=1e-5)),
    ('si x0 y el ruido tienen varianza 1, x_t también (preserva la varianza)', lambda r: _var_en_500()),
])

In [ ]:
# 3.2 Una prenda que se disuelve en ruido
_dif = usar('E4', 'difundir')
pasos_ver = [0, 50, 100, 200, 300, 500, 700, 999]
x0_demo = (X_te[Y_te == 9][:1] * 2 - 1).to(dispositivo).repeat(len(pasos_ver), 1, 1, 1)
torch.manual_seed(0)
xt_demo = _dif(x0_demo, torch.tensor(pasos_ver, device=dispositivo), torch.randn_like(x0_demo), alfa_barra)
mostrar([((x[0] + 1) / 2).clamp(0, 1).cpu().numpy() for x in xt_demo], [f't = {t}' for t in pasos_ver], tam=1.7)

### La red: adivinar el ruido
Usamos una **U-Net** pequeña (la arquitectura del tutorial 09, con bloques residuales). Recibe $x_t$, el paso $t$ y una **clase**, y predice el ruido $\hat\varepsilon$. Se entrena con un error cuadrático simple:

$$\mathcal{L} = \big\| \varepsilon - \hat\varepsilon_\theta(x_t, t, c) \big\|^2 \quad\text{con } t \text{ al azar y } x_t \text{ hecho con tu `difundir`.}$$

Dos detalles:
- **El paso $t$** entra como un vector de senos y cosenos (como el *positional encoding* del Transformer) y se suma dentro de cada bloque.
- **La clase.** El 10 % de las veces reemplazamos la clase por una etiqueta especial "sin clase" (el índice 10). Así la **misma red** aprende a generar con y sin condición. En la sección 3.4 vamos a aprovechar eso.

También guardamos una **media móvil exponencial (EMA)** de los pesos, la misma idea que el profesor de DINO en el tutorial 13. Para generar se usa la EMA, que da muestras más limpias.

In [ ]:
# 3.3 La U-Net pequeña
def emb_tiempo(t, dim=128):
    mitad = dim // 2
    frec = torch.exp(-math.log(10000) * torch.arange(mitad, device=t.device) / mitad)
    a = t.float()[:, None] * frec[None]
    return torch.cat([a.sin(), a.cos()], dim=1)

class BloqueRes(nn.Module):
    def __init__(self, cin, cout, demb):
        super().__init__()
        self.n1, self.c1 = nn.GroupNorm(8, cin), nn.Conv2d(cin, cout, 3, padding=1)
        self.emb = nn.Linear(demb, cout)
        self.n2, self.c2 = nn.GroupNorm(8, cout), nn.Conv2d(cout, cout, 3, padding=1)
        self.atajo = nn.Conv2d(cin, cout, 1) if cin != cout else nn.Identity()
    def forward(self, x, e):
        h = self.c1(F.silu(self.n1(x))) + self.emb(e)[:, :, None, None]
        return self.c2(F.silu(self.n2(h))) + self.atajo(x)

class MiniUNet(nn.Module):
    """Predice el ruido. Condicionada por el paso t y por la clase y (10 = 'sin clase')."""
    def __init__(self, c=(32, 64, 128), demb=128):
        super().__init__()
        self.t_mlp = nn.Sequential(nn.Linear(demb, demb), nn.SiLU(), nn.Linear(demb, demb))
        self.clase = nn.Embedding(11, demb)
        self.entrada = nn.Conv2d(1, c[0], 3, padding=1)
        self.b1, self.baja1 = BloqueRes(c[0], c[0], demb), nn.Conv2d(c[0], c[1], 4, 2, 1)     # 28 -> 14
        self.b2, self.baja2 = BloqueRes(c[1], c[1], demb), nn.Conv2d(c[1], c[2], 4, 2, 1)     # 14 -> 7
        self.medio1, self.medio2 = BloqueRes(c[2], c[2], demb), BloqueRes(c[2], c[2], demb)
        self.sube2, self.b3 = nn.ConvTranspose2d(c[2], c[1], 4, 2, 1), BloqueRes(2 * c[1], c[1], demb)
        self.sube1, self.b4 = nn.ConvTranspose2d(c[1], c[0], 4, 2, 1), BloqueRes(2 * c[0], c[0], demb)
        self.salida = nn.Sequential(nn.GroupNorm(8, c[0]), nn.SiLU(), nn.Conv2d(c[0], 1, 3, padding=1))
    def forward(self, x, t, y):
        e = self.t_mlp(emb_tiempo(t)) + self.clase(y)
        h1 = self.b1(self.entrada(x), e)
        h2 = self.b2(self.baja1(h1), e)
        h = self.medio2(self.medio1(self.baja2(h2), e), e)
        h = self.b3(torch.cat([self.sube2(h), h2], 1), e)       # conexiones de salto (skip) de la U-Net
        h = self.b4(torch.cat([self.sube1(h), h1], 1), e)
        return self.salida(h)

torch.manual_seed(0)
red_dif = MiniUNet().to(dispositivo)
ema_dif = copy.deepcopy(red_dif).eval().requires_grad_(False)
print(f'MiniUNet: {sum(p.numel() for p in red_dif.parameters()) / 1e6:.2f} M de parámetros')

opt = torch.optim.AdamW(red_dif.parameters(), lr=1e-3)
_dif = usar('E4', 'difundir')
hist_dif, n_pasos, t0 = [], 0, time.time()
for ep in range(CFG['ep_dif']):
    perm = torch.randperm(len(X_tr_d), device=dispositivo)
    suma, n = 0.0, 0
    for i in range(0, len(perm), 128):
        idx = perm[i:i + 128]
        x0, y = X_tr_d[idx] * 2 - 1, Y_tr_d[idx]
        t = torch.randint(0, T, (len(x0),), device=dispositivo)
        ruido = torch.randn_like(x0)
        y_in = torch.where(torch.rand(len(y), device=dispositivo) < 0.1, torch.full_like(y, 10), y)   # 10 % sin clase
        perdida = F.mse_loss(red_dif(_dif(x0, t, ruido, alfa_barra), t, y_in), ruido)
        opt.zero_grad(); perdida.backward(); opt.step()
        n_pasos += 1
        decaimiento = min(0.999, (1 + n_pasos) / (10 + n_pasos))        # EMA con "calentamiento"
        with torch.no_grad():
            for pe, p in zip(ema_dif.parameters(), red_dif.parameters()):
                pe.lerp_(p, 1 - decaimiento)
        suma += perdida.item() * len(idx); n += len(idx)
    hist_dif.append(suma / n)
    print(f'difusión época {ep + 1}/{CFG["ep_dif"]}: pérdida {hist_dif[-1]:.4f}  ({time.time() - t0:.0f} s)')

### Generar: deshacer el ruido paso a paso (DDPM)
Con la red entrenada, partimos de $x_T \sim \mathcal{N}(0, I)$ y aplicamos, para $t = T-1, \dots, 0$:

$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,\hat\varepsilon\right) + \sigma_t\, z, \qquad \sigma_t = \sqrt{\beta_t},\; z \sim \mathcal{N}(0, I),$$

y en el **último paso** ($t = 0$) no se agrega ruido. El primer término quita "la parte de ruido que la red cree ver"; el segundo vuelve a agregar un poco de azar, y eso es lo que mantiene la diversidad.

In [ ]:
# 🧩 E5 · Un paso hacia atrás (DDPM)
def paso_ddpm(x_t, eps, t, betas, alfas, alfa_barra, ruido):
    media = (x_t - betas[t] / (1 - alfa_barra[t]).sqrt() * eps) / alfas[t].sqrt()
    if t == 0:
        return media
    return media + betas[t].sqrt() * ruido

In [ ]:
# ✔️ Verificación E5
torch.manual_seed(3)
_x0 = torch.randn(4, 1, 28, 28, device=dispositivo); _e = torch.randn_like(_x0); _z = torch.randn_like(_x0)
_x1 = _REF['difundir'](_x0, torch.zeros(4, dtype=torch.long, device=dispositivo), _e, alfa_barra)
_xt = torch.randn_like(_x0)
revisar('E5', lambda: paso_ddpm(_xt, _e, 500, betas, alfas, alfa_barra, _z), [
    ('devuelve la forma de x_t', lambda r: r.shape == _xt.shape),
    ('coincide con la fórmula en t = 500', lambda r: torch.allclose(r, _REF['paso_ddpm'](_xt, _e, 500, betas, alfas, alfa_barra, _z), atol=1e-5)),
    ('en t = 0 no agrega ruido', lambda r: torch.allclose(paso_ddpm(_x1, _e, 0, betas, alfas, alfa_barra, _z), paso_ddpm(_x1, _e, 0, betas, alfas, alfa_barra, 5 * _z))),
    ('en t = 0, con el ruido verdadero, recupera x0 exactamente', lambda r: torch.allclose(paso_ddpm(_x1, _e, 0, betas, alfas, alfa_barra, _z), _x0, atol=1e-4)),
    ('el azar agregado en t = 999 tiene desviación sqrt(beta) ≈ 0,141',
     lambda r: abs(paso_ddpm(torch.zeros(64, 1, 28, 28, device=dispositivo), torch.zeros(64, 1, 28, 28, device=dispositivo), 999, betas, alfas, alfa_barra, torch.randn(64, 1, 28, 28, device=dispositivo)).std().item() - betas[999].sqrt().item()) < 0.005),
])

In [ ]:
# 3.4 El muestreador. DDPM usa tu paso_ddpm (1000 pasos). DDIM (Song et al., 2021) es determinista y salta pasos.
def x0_desde_eps(x, eps, t):
    """Despeja x0 de la fórmula de difundir: la "imagen limpia" que la red cree ver en el paso t."""
    return (x - (1 - alfa_barra[t]).sqrt() * eps) / alfa_barra[t].sqrt()

@torch.no_grad()
def predecir_ruido(modelo, x, t, clase, escala):
    tt = torch.full((len(x),), t, device=x.device, dtype=torch.long)
    vacio = torch.full((len(x),), 10, device=x.device, dtype=torch.long)
    if clase is None:
        return modelo(x, tt, vacio)
    if escala == 1.0:
        return modelo(x, tt, clase)
    e_incond, e_cond = modelo(torch.cat([x, x]), torch.cat([tt, tt]), torch.cat([vacio, clase])).chunk(2)
    return usar('E6', 'guiar')(e_incond, e_cond, escala)      # E6 viene en la sección siguiente

@torch.no_grad()
def muestrear(modelo, n, clase=None, escala=1.0, pasos=None, semilla=0, capturar=()):
    """pasos=None -> DDPM con los 1000 pasos. pasos=k -> DDIM con k pasos.
    clase: None (incondicional) o un entero 0-9. capturar: pasos t en los que guardar (x_t, x̂0).
    Devuelve (imágenes en [0,1], dict t -> (x_t, x̂0))."""
    g = torch.Generator(device=dispositivo).manual_seed(semilla)
    x = torch.randn(n, 1, 28, 28, device=dispositivo, generator=g)
    y = None if clase is None else torch.full((n,), clase, device=dispositivo, dtype=torch.long)
    pelicula = {}
    if pasos is None:
        paso = usar('E5', 'paso_ddpm')
        for t in range(T - 1, -1, -1):
            eps = predecir_ruido(modelo, x, t, y, escala)
            if t in capturar:
                pelicula[t] = (x.cpu(), x0_desde_eps(x, eps, t).cpu())
            x = paso(x, eps, t, betas, alfas, alfa_barra, torch.randn(x.shape, device=dispositivo, generator=g))
    else:
        ts = torch.linspace(T - 1, 0, pasos).round().long().tolist()
        for i, t in enumerate(ts):
            eps = predecir_ruido(modelo, x, t, y, escala)
            x0 = x0_desde_eps(x, eps, t).clamp(-1, 1)
            eps = (x - alfa_barra[t].sqrt() * x0) / (1 - alfa_barra[t]).sqrt()
            if t in capturar:
                pelicula[t] = (x.cpu(), x0.cpu())
            ab_sig = alfa_barra[ts[i + 1]] if i + 1 < len(ts) else torch.tensor(1.0, device=dispositivo)
            x = ab_sig.sqrt() * x0 + (1 - ab_sig).sqrt() * eps
    return ((x.clamp(-1, 1) + 1) / 2).cpu(), pelicula

def muestrear_muchas(modelo, n, **kw):
    partes = [muestrear(modelo, min(CFG['lote_gen'], n - i), semilla=i, **kw)[0] for i in range(0, n, CFG['lote_gen'])]
    return torch.cat(partes)

# La película de la generación: arriba x_t, abajo lo que la red cree que es la imagen final (x̂0)
pasos_peli = [999, 900, 700, 500, 300, 150, 50, 0]
t0 = time.time()
muestras_ddpm, peli = muestrear(ema_dif, 16, pasos=None, capturar=pasos_peli)
print(f'DDPM, 1000 pasos: {time.time() - t0:.1f} s para 16 prendas')
fig, ax = plt.subplots(4, len(pasos_peli), figsize=(1.6 * len(pasos_peli), 6.8))
for j, t in enumerate(pasos_peli):
    for fila, (k, nombre) in enumerate([(0, 'x_t'), (1, 'x̂0'), (0, 'x_t'), (1, 'x̂0')]):
        img = peli[t][k][0 if fila < 2 else 1, 0]
        ax[fila, j].imshow(((img + 1) / 2).clamp(0, 1), cmap='gray', vmin=0, vmax=1); ax[fila, j].axis('off')
        if fila == 0: ax[fila, j].set_title(f't = {t}', fontsize=9)
for fila, nombre in enumerate(['x_t', 'x̂0', 'x_t', 'x̂0']):
    ax[fila, 0].text(-4, 14, nombre, ha='right', va='center', fontsize=10)
plt.suptitle('Dos prendas naciendo del ruido'); plt.tight_layout(); plt.show()

In [ ]:
# DDPM (1000 pasos) vs DDIM (50 pasos), sin clase
t0 = time.time(); muestras_ddim, _ = muestrear(ema_dif, 64, pasos=50); t_ddim = time.time() - t0
t0 = time.time(); muestras_ddpm, _ = muestrear(ema_dif, 64, pasos=None); t_ddpm = time.time() - t0
fig, ax = plt.subplots(1, 2, figsize=(11, 5.5))
ax[0].imshow(grilla(muestras_ddpm, 8), cmap='gray'); ax[0].axis('off'); ax[0].set_title(f'DDPM · 1000 pasos · {t_ddpm:.1f} s')
ax[1].imshow(grilla(muestras_ddim, 8), cmap='gray'); ax[1].axis('off'); ax[1].set_title(f'DDIM · 50 pasos · {t_ddim:.1f} s')
plt.tight_layout(); plt.show()

### Guía libre de clasificador (*classifier-free guidance*)
Nuestra red sabe predecir el ruido **con** clase ($\hat\varepsilon_c$) y **sin** clase ($\hat\varepsilon_\varnothing$). La guía mezcla las dos y exagera la diferencia:

$$\tilde\varepsilon = \hat\varepsilon_\varnothing + s\,\big(\hat\varepsilon_c - \hat\varepsilon_\varnothing\big)$$

Con $s = 0$ se ignora la clase; con $s = 1$ es la predicción condicional normal; con $s > 1$ se empuja la muestra "más hacia la clase" de lo que la red haría por sí sola. **Es exactamente el parámetro `guidance_scale` de Stable Diffusion** (sección 4), con la misma convención.

In [ ]:
# 🧩 E6 · Guía libre de clasificador
def guiar(eps_incond, eps_cond, escala):
    return eps_incond + escala * (eps_cond - eps_incond)

In [ ]:
# ✔️ Verificación E6
_a, _b = torch.randn(3, 1, 4, 4), torch.randn(3, 1, 4, 4)
revisar('E6', lambda: guiar(_a, _b, 7.5), [
    ('escala 0 -> predicción sin clase', lambda r: torch.allclose(guiar(_a, _b, 0.0), _a)),
    ('escala 1 -> predicción con clase', lambda r: torch.allclose(guiar(_a, _b, 1.0), _b)),
    ('escala 7,5 -> coincide con la fórmula', lambda r: torch.allclose(r, _REF['guiar'](_a, _b, 7.5))),
    ('mantiene la forma', lambda r: r.shape == _a.shape),
])

In [ ]:
# 3.5 Pídele un botín con distintas escalas de guía
ESCALAS = [0.0, 1.0, 3.0, 7.0]
fig, ax = plt.subplots(1, len(ESCALAS), figsize=(4 * len(ESCALAS), 4.2))
for a, s in zip(ax, ESCALAS):
    m, _ = muestrear(ema_dif, 16, clase=9, escala=s, pasos=50, semilla=1)
    a.imshow(grilla(m, 4), cmap='gray'); a.axis('off'); a.set_title(f'clase "botín", escala = {s:g}')
plt.tight_layout(); plt.show()

# Ahora con números, para las 10 clases: ¿el juez reconoce la clase pedida? ¿cuán distintas son entre sí?
def diversidad(feats):
    f = F.normalize(feats, dim=1)
    return (1 - f @ f.T).sum().item() / (len(f) * (len(f) - 1))      # distancia coseno media entre pares

tabla_guia = {}
for s in ESCALAS:
    aciertos, divs = [], []
    for c in range(10):
        m, _ = muestrear(ema_dif, CFG['n_guia'], clase=c, escala=s, pasos=50, semilla=100 + c)
        P, Fs = opinar(m)
        aciertos.append((P.argmax(1) == c).float().mean().item()); divs.append(diversidad(Fs))
    tabla_guia[s] = (np.mean(aciertos), np.mean(divs))
    print(f'escala {s:>4g}: el juez reconoce la clase pedida en {tabla_guia[s][0]:.1%} · diversidad {tabla_guia[s][1]:.3f}')

fig, ax1 = plt.subplots(figsize=(6, 3.5)); ax2 = ax1.twinx()
ax1.plot(ESCALAS, [v[0] for v in tabla_guia.values()], 'o-', color='C0'); ax1.set_ylabel('clase correcta según el juez', color='C0')
ax2.plot(ESCALAS, [v[1] for v in tabla_guia.values()], 's--', color='C3'); ax2.set_ylabel('diversidad dentro de la clase', color='C3')
ax1.set_xlabel('escala de guía s'); plt.title('Guía: fidelidad vs diversidad'); plt.tight_layout(); plt.show()

Más guía = más fidelidad a lo pedido, a cambio de **menos variedad** (y, si te pasas, de imágenes saturadas o "caricaturizadas"). Ese mismo compromiso lo vas a ver en Stable Diffusion.

### ¿Cómo comparamos generadores? La distancia de Fréchet
"Se ve bien" no es una métrica. La más usada es **FID** (*Fréchet Inception Distance*): se pasan las imágenes reales y las generadas por una red ya entrenada, se modela cada nube de *features* como una gaussiana (media $\mu$, covarianza $\Sigma$), y se mide la distancia entre las dos gaussianas:

$$d^2 = \|\mu_1 - \mu_2\|^2 + \operatorname{Tr}\!\left(\Sigma_1 + \Sigma_2 - 2\,(\Sigma_1\Sigma_2)^{1/2}\right)$$

Castiga dos cosas a la vez: imágenes que no parecen reales (las medias se separan) y falta de variedad (las covarianzas no calzan). FID usa la red Inception-v3; nosotros usaremos las *features* de 128 dimensiones de nuestro juez ("FID-lite"). Los números no son comparables con los de los papers, pero sí entre nuestros modelos.

In [ ]:
# 🧩 E7 · Distancia de Fréchet
from scipy import linalg
def distancia_frechet(f1, f2):
    f1, f2 = np.asarray(f1, dtype=np.float64), np.asarray(f2, dtype=np.float64)
    m1, m2 = f1.mean(0), f2.mean(0)
    s1, s2 = np.cov(f1, rowvar=False), np.cov(f2, rowvar=False)
    raiz = linalg.sqrtm(s1 @ s2)
    if isinstance(raiz, tuple):
        raiz = raiz[0]
    return float(((m1 - m2) ** 2).sum() + np.trace(s1 + s2 - 2 * raiz.real))

In [ ]:
# ✔️ Verificación E7
_rng = np.random.default_rng(0)
_f = _rng.normal(size=(500, 6)) @ _rng.normal(size=(6, 6)) + 1.0
_d = np.array([1.0, -2.0, 0.0, 0.5, 0.0, 0.0])
revisar('E7', lambda: distancia_frechet(_f, _f), [
    ('un conjunto contra sí mismo da ≈ 0', lambda r: abs(r) < 1e-4),
    ('trasladar todo por un vector d da |d|² = 5,25', lambda r: abs(distancia_frechet(_f, _f + _d) - 5.25) < 1e-4),
    ('duplicar todas las features da |mu|² + Tr(Sigma)',
     lambda r: abs(distancia_frechet(_f, 2 * _f) - ((_f.mean(0) ** 2).sum() + np.trace(np.cov(_f, rowvar=False)))) < 1e-3),
    ('es simétrica', lambda r: abs(distancia_frechet(_f[:250], _f[250:] + 0.3) - distancia_frechet(_f[250:] + 0.3, _f[:250])) < 1e-4),
    ('acepta tensores de torch', lambda r: abs(distancia_frechet(torch.tensor(_f), torch.tensor(_f + _d)) - 5.25) < 1e-4),
])

In [ ]:
# 3.6 Los cuatro generadores frente a frente
frechet = usar('E7', 'distancia_frechet')
N = CFG['n_eval']
_, F_real = opinar(X_te[: N])
_, F_real2 = opinar(X_te[N: 2 * N] if len(X_te) >= 2 * N else X_te[-N:])

@torch.no_grad()
def gen_ae(n):  return torch.sigmoid(ae.dec(torch.distributions.MultivariateNormal(media_ae, cov_ae).sample((n,)).to(dispositivo))).cpu()
@torch.no_grad()
def gen_vae(n): return torch.sigmoid(vae.dec(torch.randn(n, 2, device=dispositivo))).cpu()
@torch.no_grad()
def gen_gan(n): return ((G(torch.randn(n, Z_GAN, device=dispositivo)) + 1) / 2).cpu()

generadores = {
    'Autoencoder': gen_ae, 'VAE': gen_vae, 'GAN': gen_gan,
    'Difusión · DDIM 50': lambda n: muestrear_muchas(ema_dif, n, pasos=50),
    'Difusión · DDPM 1000': lambda n: muestrear_muchas(ema_dif, n, pasos=None),
}
COMPARACION, MUESTRAS = {}, {}
print(f'{"modelo":22s} {"FID-lite ↓":>11s} {"confianza ↑":>12s} {"cobertura ↑":>12s} {"s / 100 muestras":>17s}')
print(f'{"(prueba vs prueba)":22s} {frechet(F_real, F_real2):11.2f}')
for nombre, gen in generadores.items():
    if GPU: torch.cuda.synchronize()
    t0 = time.time(); m = gen(N)
    if GPU: torch.cuda.synchronize()
    seg = (time.time() - t0) * 100 / N
    _, Fg = opinar(m); r = resumen_juez(m)
    COMPARACION[nombre] = dict(fid=frechet(F_real, Fg), **r, seg=seg); MUESTRAS[nombre] = m[:12]
    c = COMPARACION[nombre]
    print(f'{nombre:22s} {c["fid"]:11.2f} {c["confianza"]:12.3f} {c["cobertura"]:12.3f} {c["seg"]:17.3f}')

fig, ax = plt.subplots(len(MUESTRAS), 1, figsize=(11, 1.05 * len(MUESTRAS)))
for a, (nombre, m) in zip(ax, MUESTRAS.items()):
    a.imshow(grilla(m, 1), cmap='gray'); a.set_xticks([]); a.set_yticks([]); a.set_ylabel(nombre, rotation=0, ha='right', va='center', fontsize=9)
plt.tight_layout(); plt.show()
for _n in ['red_dif', 'opt']:              # ya no los necesitamos (para generar usamos ema_dif)
    globals().pop(_n, None)
memoria_gpu('al terminar la parte A')

**Para conversar: el trilema de los modelos generativos.**
- **Calidad:** la GAN y la difusión producen prendas nítidas y quedan muy por delante del autoencoder y el VAE en FID-lite. Con la configuración completa (GPU), en nuestra corrida de referencia quedaron prácticamente empatadas.
- **Diversidad / cobertura:** aquí las dos cubren las 10 prendas. En problemas más grandes, la GAN es la que tiende a "olvidar" modos.
- **Velocidad:** el VAE y la GAN generan con *una* pasada por la red; la difusión necesita decenas o cientos. Mira la última columna: la diferencia es de varios órdenes de magnitud.

Entonces, ¿por qué ganó la difusión si en Fashion-MNIST empata con la GAN? Por lo que no sale en la tabla: entrenarla es tan **estable** como entrenar un VAE (un simple error cuadrático que baja sin sobresaltos, sin un juego entre dos redes), **escala** bien con más datos y modelos más grandes, y se **condiciona** con facilidad, como viste con la guía. El precio es la velocidad, y por eso buena parte de la investigación reciente busca muestrear en menos pasos (DDIM, *distillation*, *consistency models*, *flow matching*).

> **El salto a Stable Diffusion.** Hacer difusión sobre píxeles de 512×512 es carísimo. La solución (*latent diffusion*, Rombach et al., 2022) fue combinar lo que ya viste: **un VAE** comprime la imagen, **la difusión** ocurre en ese espacio comprimido, y la **condición** ya no es una de 10 clases sino una frase cualquiera, codificada por **CLIP**.

---
### ☕ Pausa sugerida (10 min)

## 4 · Stable Diffusion por dentro (25 min)

Stable Diffusion 1.5 tiene exactamente las piezas que ya construiste, en grande:

| Pieza | En Fashion-MNIST (secciones 1–3) | En Stable Diffusion |
|---|---|---|
| Espacio donde ocurre la difusión | píxeles 28×28 | **latente del VAE**: 4×64×64 para una imagen de 512×512 |
| Red que predice el ruido | `MiniUNet` (≈1 M parámetros) | U-Net con atención (≈860 M) |
| Condición | una de 10 clases (`nn.Embedding`) | una frase, codificada por **el codificador de texto de CLIP** (77 vectores de 768) y leída con **atención cruzada** |
| Guía | `guiar` (E6) | `guidance_scale`, la misma fórmula |
| Muestreador | DDPM / DDIM | DDIM (y otros) |

Vamos a abrir el *pipeline* de `diffusers`, mirar cada pieza y después **escribir el bucle de generación a mano**.

> **Notas.** El repositorio original `runwayml/stable-diffusion-v1-5` fue retirado de Hugging Face; usamos la copia oficial `stable-diffusion-v1-5/stable-diffusion-v1-5`. La licencia es CreativeML OpenRAIL-M, que prohíbe ciertos usos; el *pipeline* trae activado un **filtro de seguridad** que entrega una imagen negra si detecta contenido inapropiado. Lo dejamos activado. La descarga pesa ≈2,5 GB en GPU (pesos en float16).

In [ ]:
# 4.1 Cargar Stable Diffusion y mirar sus piezas
from diffusers import StableDiffusionPipeline, StableDiffusionImg2ImgPipeline, DDIMScheduler
SD_ID = 'stable-diffusion-v1-5/stable-diffusion-v1-5'
dtype_sd = torch.float16 if GPU else torch.float32
try:
    pipe = StableDiffusionPipeline.from_pretrained(SD_ID, torch_dtype=dtype_sd, variant='fp16' if GPU else None)
except Exception:
    pipe = StableDiffusionPipeline.from_pretrained(SD_ID, torch_dtype=dtype_sd)
pipe = pipe.to(dispositivo)
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config, clip_sample=False)   # el muestreador de la sección 3
pipe.set_progress_bar_config(disable=True)

partes_sd = {'VAE (comprime y descomprime)': pipe.vae, 'U-Net (predice el ruido)': pipe.unet, 'Codificador de texto (CLIP)': pipe.text_encoder}
if getattr(pipe, 'safety_checker', None) is not None:
    partes_sd['Filtro de seguridad'] = pipe.safety_checker
for nombre, m in partes_sd.items():
    print(f'{nombre:32s} {sum(p.numel() for p in m.parameters()) / 1e6:8.1f} M parámetros')
LADO = CFG['sd_lado']
print(f'\nImágenes de {LADO}×{LADO}; latentes de 4×{LADO // 8}×{LADO // 8}; factor de escala del VAE = {pipe.vae.config.scaling_factor}')
memoria_gpu('con Stable Diffusion cargado')

In [ ]:
# 4.2 El VAE de Stable Diffusion: 48 veces menos números, casi la misma imagen
def a_cuadrado(im, lado):
    w, h = im.size; s = min(w, h)
    return im.crop(((w - s) // 2, (h - s) // 2, (w - s) // 2 + s, (h - s) // 2 + s)).resize((lado, lado), Image.BICUBIC)

def a_tensor_sd(im):
    """PIL -> (1,3,H,W) en [-1,1], en el dispositivo y con el tipo de SD."""
    return torch.tensor(np.array(im)).permute(2, 0, 1)[None].float().div(127.5).sub(1).to(dispositivo, dtype_sd)

def a_pil(x):
    """(3,H,W) o (1,3,H,W) en [0,1] -> PIL."""
    x = x[0] if x.dim() == 4 else x
    return Image.fromarray((x.float().clamp(0, 1).permute(1, 2, 0).cpu().numpy() * 255).round().astype(np.uint8))

jirafa = a_cuadrado(foto(FOTO_JIRAFA), LADO)
with torch.no_grad():
    lat = pipe.vae.encode(a_tensor_sd(jirafa)).latent_dist.mean * pipe.vae.config.scaling_factor
    rec = pipe.vae.decode(lat / pipe.vae.config.scaling_factor).sample
rec01 = (rec.float() / 2 + 0.5).clamp(0, 1)
orig01 = torch.tensor(np.array(jirafa)).permute(2, 0, 1)[None].float().div(255).to(dispositivo)
psnr = -10 * torch.log10(F.mse_loss(rec01, orig01)).item()
print(f'Imagen: {tuple(orig01.shape[1:])} = {orig01[0].numel():,} números · latente: {tuple(lat.shape[1:])} = {lat[0].numel():,} números '
      f'· compresión ×{orig01[0].numel() / lat[0].numel():.0f} · PSNR de la reconstrucción: {psnr:.1f} dB')
canales = [((c - c.min()) / (c.max() - c.min() + 1e-8)).float().cpu().numpy() for c in lat[0]]
mostrar([jirafa] + canales + [a_pil(rec01)], ['original'] + [f'latente, canal {i}' for i in range(4)] + ['reconstrucción'], tam=2.6)

Los 4 canales del latente todavía "se parecen" a la foto: el VAE conserva la estructura espacial y descarta detalle fino que el decodificador sabe rellenar. Por eso la difusión puede trabajar en ese espacio **48 veces más pequeño** sin perder mucho: es lo que hizo posible generar imágenes de 512×512 en una GPU común.

In [ ]:
# 4.3 El texto: 77 tokens -> 77 vectores que la U-Net lee con atención cruzada
PROMPT = 'a watercolor painting of a fox reading a book in a cozy library, warm light'
tok = pipe.tokenizer(PROMPT, padding='max_length', max_length=pipe.tokenizer.model_max_length, truncation=True, return_tensors='pt')
n_reales = int(tok.attention_mask.sum())
print(f'{n_reales} tokens reales + {pipe.tokenizer.model_max_length - n_reales} de relleno = {pipe.tokenizer.model_max_length}')
print(pipe.tokenizer.convert_ids_to_tokens(tok.input_ids[0, :n_reales].tolist()))
with torch.no_grad():
    emb_txt = pipe.text_encoder(tok.input_ids.to(dispositivo))[0]
print('Embedding del texto:', tuple(emb_txt.shape), '-> lo que la U-Net "lee" en cada paso')

### El bucle de generación, a mano
Esto es lo que hace `pipe(prompt)` por dentro. Ya conoces cada paso:
1. Codificar **dos** textos: el vacío `""` (para la predicción sin condición) y el prompt.
2. Crear el latente inicial: ruido gaussiano de forma $(1, 4, \text{lado}/8, \text{lado}/8)$, multiplicado por `scheduler.init_noise_sigma`.
3. Para cada paso $t$ del muestreador: pasar **dos copias** del latente por la U-Net (una con cada texto), separar las dos predicciones, combinarlas con **tu** `guiar` y dar un paso con `scheduler.step`.
4. Decodificar con el VAE (dividiendo primero por el factor de escala) y llevar de $[-1, 1]$ a $[0, 1]$.

In [ ]:
# 🧩 E8 · Stable Diffusion a mano
@torch.no_grad()
def generar_a_mano(prompt, pasos=25, escala=7.5, semilla=0, lado=CFG['sd_lado']):
    ids = pipe.tokenizer(['', prompt], padding='max_length', max_length=pipe.tokenizer.model_max_length,
                         truncation=True, return_tensors='pt').input_ids.to(dispositivo)
    textos = pipe.text_encoder(ids)[0]
    g = torch.Generator(device=dispositivo).manual_seed(semilla)
    lat = torch.randn((1, 4, lado // 8, lado // 8), generator=g, device=dispositivo, dtype=textos.dtype)
    lat = lat * pipe.scheduler.init_noise_sigma
    pipe.scheduler.set_timesteps(pasos, device=dispositivo)
    for t in pipe.scheduler.timesteps:
        entrada = pipe.scheduler.scale_model_input(torch.cat([lat, lat]), t)
        eps = pipe.unet(entrada, t, encoder_hidden_states=textos).sample
        eps_incond, eps_cond = eps.chunk(2)
        eps = guiar(eps_incond, eps_cond, escala)
        lat = pipe.scheduler.step(eps, t, lat).prev_sample
    img = pipe.vae.decode(lat / pipe.vae.config.scaling_factor).sample
    return (img / 2 + 0.5).clamp(0, 1)[0]

In [ ]:
# ✔️ Verificación E8 (compara contra el pipeline oficial con la misma semilla)
PRUEBA_SD = dict(prompt='a product photo of a leather ankle boot on a white background', pasos=CFG['sd_pasos_prueba'], escala=7.5, semilla=1)
def _con_pipe(prompt, pasos, escala, semilla):
    g = torch.Generator(device=dispositivo).manual_seed(semilla)
    return pipe(prompt, num_inference_steps=pasos, guidance_scale=escala, generator=g, height=LADO, width=LADO, output_type='pt').images[0]
_ref_pipe = _con_pipe(**PRUEBA_SD).float().cpu()
revisar('E8', lambda: generar_a_mano(**PRUEBA_SD), [
    (f'devuelve un tensor (3, {LADO}, {LADO})', lambda r: tuple(r.shape) == (3, LADO, LADO)),
    ('valores en [0, 1]', lambda r: r.min() >= 0 and r.max() <= 1),
    ('coincide con pipe(...) usando la misma semilla', lambda r: (r.float().cpu() - _ref_pipe).abs().max() < (0.03 if GPU else 1e-3)),
])
if funciona('E8'):
    mostrar([a_pil(_ref_pipe), a_pil(generar_a_mano(**PRUEBA_SD))], ['pipe(...)', 'generar_a_mano(...)'], tam=3.5, sup=PRUEBA_SD['prompt'])

In [ ]:
# 4.4 La película: el latente decodificado a lo largo de los pasos
capturas = []
def _guardar(p, i, t, kw):
    capturas.append((i, kw['latents'].detach().clone()))
    return kw
g = torch.Generator(device=dispositivo).manual_seed(0)
final = pipe(PROMPT, num_inference_steps=CFG['sd_pasos'], guidance_scale=7.5, generator=g, height=LADO, width=LADO,
             callback_on_step_end=_guardar, callback_on_step_end_tensor_inputs=['latents']).images[0]
elegidos = sorted({0, len(capturas) // 4, len(capturas) // 2, 3 * len(capturas) // 4, len(capturas) - 1})
cuadros = []
with torch.no_grad():
    for i in elegidos:
        cuadros.append(a_pil((pipe.vae.decode(capturas[i][1] / pipe.vae.config.scaling_factor).sample / 2 + 0.5).clamp(0, 1)))
mostrar(cuadros, [f'paso {i + 1}/{len(capturas)}' for i in elegidos], tam=3.0, sup=PROMPT)

In [ ]:
# 4.5 La escala de guía y la semilla
PROMPT_GUIA = 'a product photo of a leather ankle boot on a white background'
ESCALAS_SD = [1.0, 3.0, 7.5, 15.0]
IMAGENES_GUIA = {}
for s in ESCALAS_SD:
    g = torch.Generator(device=dispositivo).manual_seed(0)
    IMAGENES_GUIA[s] = pipe(PROMPT_GUIA, num_inference_steps=CFG['sd_pasos'], guidance_scale=s, generator=g, height=LADO, width=LADO).images[0]
semillas = []
for sem in [1, 2, 3, 4]:
    g = torch.Generator(device=dispositivo).manual_seed(sem)
    semillas.append(pipe(PROMPT_GUIA, num_inference_steps=CFG['sd_pasos'], guidance_scale=7.5, generator=g, height=LADO, width=LADO).images[0])
mostrar(list(IMAGENES_GUIA.values()), [f'escala {s:g} · semilla 0' for s in ESCALAS_SD], tam=3.2, sup=PROMPT_GUIA)
mostrar(semillas, [f'escala 7,5 · semilla {s}' for s in [1, 2, 3, 4]], tam=3.2)

Igual que con los botines de la sección 3: con escala 1 la imagen sigue el texto "a medias"; alrededor de 7,5 obedece bien; con 15 los colores se saturan y todo se vuelve más "de catálogo". La **semilla** fija el ruido inicial: misma semilla y mismo prompt dan la misma imagen; otra semilla, otra composición.

### Imagen a imagen: el proceso hacia adelante como herramienta
Si en vez de partir de ruido puro tomamos una foto, la llevamos con **`difundir`** hasta un paso intermedio y desde ahí deshacemos el ruido **con otro prompt**, obtenemos una versión de la foto guiada por el texto (SDEdit, Meng et al., 2022). El parámetro `strength` dice hasta dónde se avanza en el proceso hacia adelante: con 0,3 se conserva casi todo; con 0,9 queda poco más que la composición.

In [ ]:
# 4.6 Imagen a imagen sobre el gato de COCO
i2i = StableDiffusionImg2ImgPipeline(**pipe.components)
i2i.set_progress_bar_config(disable=True)
gato_sd = a_cuadrado(foto(FOTO_GATO), LADO)
PROMPT_I2I = 'a watercolor painting of a cat sitting on a car'
salidas = []
for fuerza in [0.3, 0.6, 0.9]:
    g = torch.Generator(device=dispositivo).manual_seed(0)
    salidas.append(i2i(PROMPT_I2I, image=gato_sd, strength=fuerza, guidance_scale=7.5,
                       num_inference_steps=max(CFG['sd_pasos'], 10), generator=g).images[0])
mostrar([gato_sd] + salidas, ['foto original', 'strength 0,3', 'strength 0,6', 'strength 0,9'], tam=3.2, sup=PROMPT_I2I)

**Lo que quedó fuera (y hacia dónde fue el campo).** SDXL y SD3 escalaron la misma receta. Los modelos más recientes (SD3, FLUX) reemplazan la U-Net por un **Transformer de difusión** (DiT, el ViT del tutorial 12 aplicado a parches del latente) y la difusión por *flow matching*, que permite generar en menos pasos. Los generadores de **video** (Sora, Veo, Wan) aplican la misma idea a latentes espacio-temporales.

## 5 · CLIP: imágenes y frases en el mismo espacio (30 min)

**CLIP** (Radford et al., 2021) tiene dos codificadores, uno de imágenes (un ViT) y uno de texto (un Transformer), que proyectan a un **mismo espacio** de 512 dimensiones. Se entrenó con 400 millones de pares (imagen, texto alternativo) sacados de internet, con un objetivo **contrastivo**: en un lote de $N$ pares, la imagen $i$ tiene que parecerse a su texto $i$ y no a los otros $N-1$.

Es el InfoNCE del tutorial 13, pero entre **dos modalidades**: se calcula la matriz $N\times N$ de similitudes coseno, se multiplica por una temperatura aprendida ($\approx 100$) y se aplica entropía cruzada **por filas** (cada imagen elige su texto) y **por columnas** (cada texto elige su imagen). La pérdida es el promedio de las dos.

Usamos `openai/clip-vit-base-patch32`, el mismo tipo de codificador de texto que acabas de ver dentro de Stable Diffusion.

In [ ]:
# 5.1 Cargar CLIP y dos funciones para obtener embeddings
# Antes, Stable Diffusion se va a la RAM hasta el cierre (sección 8): en una T4 (15 GB) no caben con holgura
# SD, CLIP, LLaVA y SAM a la vez, sobre todo cuando SAM procesa muchos prompts.
globals().pop('i2i', None)
if 'pipe' in globals():
    pipe.to('cpu')
memoria_gpu('con Stable Diffusion en la RAM')

from transformers import CLIPModel, CLIPProcessor
CLIP_ID = 'openai/clip-vit-base-patch32'
clip = CLIPModel.from_pretrained(CLIP_ID).to(dispositivo).eval()
proc_clip = CLIPProcessor.from_pretrained(CLIP_ID)
ESCALA_CLIP = clip.logit_scale.exp().item()

def _vector(salida):
    return salida if torch.is_tensor(salida) else salida.pooler_output   # transformers ≥ 5 devuelve un objeto

@torch.no_grad()
def emb_imagenes(imagenes, lote=64):
    """Lista de imágenes PIL -> tensor (N, 512) en CPU, SIN normalizar."""
    salida = []
    for i in range(0, len(imagenes), lote):
        px = proc_clip(images=imagenes[i:i + lote], return_tensors='pt')['pixel_values'].to(dispositivo)
        salida.append(_vector(clip.get_image_features(pixel_values=px)).float().cpu())
    return torch.cat(salida)

@torch.no_grad()
def emb_textos(textos, lote=256):
    """Lista de frases -> tensor (N, 512) en CPU, SIN normalizar."""
    salida = []
    for i in range(0, len(textos), lote):
        tk = proc_clip(text=textos[i:i + lote], return_tensors='pt', padding=True, truncation=True).to(dispositivo)
        salida.append(_vector(clip.get_text_features(input_ids=tk['input_ids'], attention_mask=tk['attention_mask'])).float().cpu())
    return torch.cat(salida)

print(f'CLIP: {sum(p.numel() for p in clip.parameters()) / 1e6:.0f} M parámetros · temperatura (logit_scale) = {ESCALA_CLIP:.1f}')

# Datos para esta sección: CIFAR-10 e Imagenette (los del tutorial 08 y la Tarea 2) + Fashion-MNIST
IMAGENETTE = {'n01440764': 'tench', 'n02102040': 'English springer', 'n02979186': 'cassette player', 'n03000684': 'chain saw',
              'n03028079': 'church', 'n03394916': 'French horn', 'n03417042': 'garbage truck', 'n03425413': 'gas pump',
              'n03445777': 'golf ball', 'n03888257': 'parachute'}
cifar_im, cifar_y, CIFAR = leer_carpeta(ZIPS['cifar10'], 'cifar10/test', CFG['clip_k'])
inet_im, inet_y, _wn = leer_carpeta(ZIPS['imagenette160'], 'imagenette160/val', CFG['clip_k'])
idx_f = torch.cat([torch.nonzero(Y_te == c)[:CFG['clip_k'], 0] for c in range(10)])
fash_im = [Image.fromarray((X_te[i, 0].numpy() * 255).astype(np.uint8)).convert('RGB') for i in idx_f]
CONJUNTOS = {'CIFAR-10': (cifar_im, cifar_y, CIFAR), 'Imagenette': (inet_im, inet_y, [IMAGENETTE[w] for w in _wn]),
             'Fashion-MNIST': (fash_im, Y_te[idx_f].numpy(), PRENDAS_EN)}
t0 = time.time()
EMB = {k: emb_imagenes(v[0]) for k, v in CONJUNTOS.items()}
print(f'Embeddings de {sum(len(v[0]) for v in CONJUNTOS.values())} imágenes en {time.time() - t0:.0f} s')

In [ ]:
# 🧩 E9 · La pérdida contrastiva de CLIP
def perdida_clip(emb_img, emb_txt, escala):
    i, t = F.normalize(emb_img, dim=-1), F.normalize(emb_txt, dim=-1)
    logits = escala * i @ t.T
    etiquetas = torch.arange(len(i), device=logits.device)
    return (F.cross_entropy(logits, etiquetas) + F.cross_entropy(logits.T, etiquetas)) / 2

In [ ]:
# ✔️ Verificación E9: comparamos con la pérdida que calcula el propio CLIPModel
idx8 = [int(np.nonzero(cifar_y == c)[0][0]) for c in range(8)]
im8, txt8 = [cifar_im[i] for i in idx8], [f'a photo of a {CIFAR[c]}.' for c in range(8)]
with torch.no_grad():
    _ent = proc_clip(text=txt8, images=im8, return_tensors='pt', padding=True).to(dispositivo)
    _perdida_hf = clip(**_ent, return_loss=True).loss.item()
EI8, ET8 = emb_imagenes(im8), emb_textos(txt8)
revisar('E9', lambda: perdida_clip(EI8, ET8, ESCALA_CLIP), [
    ('devuelve un escalar', lambda r: torch.as_tensor(r).numel() == 1),
    (f'coincide con CLIPModel(..., return_loss=True) = {_perdida_hf:.4f}', lambda r: abs(float(r) - _perdida_hf) < 1e-3 * max(1, _perdida_hf)),
    ('es simétrica: intercambiar imágenes y textos da lo mismo', lambda r: abs(float(perdida_clip(ET8, EI8, ESCALA_CLIP)) - float(r)) < 1e-5),
    ('con pares perfectos y ortogonales la pérdida es ≈ 0', lambda r: float(perdida_clip(torch.eye(8), 3 * torch.eye(8), 100.0)) < 1e-6),
    ('si todo es igual (sin información) vale log N', lambda r: abs(float(perdida_clip(torch.ones(8, 4), torch.ones(8, 4), 100.0)) - math.log(8)) < 1e-4),
])

S = F.normalize(EI8, dim=-1) @ F.normalize(ET8, dim=-1).T
fig, ax = plt.subplots(1, 2, figsize=(13, 4.3), gridspec_kw=dict(width_ratios=[1.6, 1]))
ax[0].imshow(np.concatenate([np.array(im.resize((64, 64))) for im in im8], axis=1)); ax[0].axis('off'); ax[0].set_title('8 imágenes de CIFAR-10')
ax[1].imshow(S, cmap='viridis'); ax[1].set_xticks(range(8)); ax[1].set_xticklabels([CIFAR[c] for c in range(8)], rotation=60, fontsize=8)
ax[1].set_yticks(range(8)); ax[1].set_yticklabels([f'imagen {i}' for i in range(8)], fontsize=8)
for i in range(8):
    for j in range(8):
        ax[1].text(j, i, f'{S[i, j]:.2f}', ha='center', va='center', fontsize=6, color='w')
ax[1].set_title('Similitud coseno imagen × texto'); plt.tight_layout(); plt.show()

Fíjate en la escala de colores: las similitudes coseno de CLIP viven en un rango estrecho (más o menos entre 0,05 y 0,3), y la diagonal gana por poco. Por eso hace falta la **temperatura** ≈ 100: convierte diferencias de centésimas en probabilidades claras.

### Clasificar sin entrenar (*zero-shot*)
Para clasificar entre $C$ clases: se escribe una frase por clase ("a photo of a dog."), se calculan sus embeddings y cada imagen elige la frase más parecida. **No se entrena nada**: el "clasificador" son los embeddings de texto.

In [ ]:
# 🧩 E10 · Clasificación zero-shot
def zero_shot(emb_img, emb_txt, escala=100.0):
    i, t = F.normalize(emb_img.float(), dim=-1), F.normalize(emb_txt.float(), dim=-1)
    return (escala * i @ t.T).softmax(dim=-1)

In [ ]:
# ✔️ Verificación E10
_ti = emb_textos([f'a photo of a {c}.' for c in CIFAR])
_ii = EMB['CIFAR-10'][:50]
revisar('E10', lambda: zero_shot(_ii, _ti, ESCALA_CLIP), [
    ('forma (N, C)', lambda r: tuple(r.shape) == (50, 10)),
    ('cada fila suma 1', lambda r: torch.allclose(r.sum(1), torch.ones(50), atol=1e-4)),
    ('no depende del largo de los embeddings (normaliza)', lambda r: torch.allclose(zero_shot(3 * _ii, 0.5 * _ti, ESCALA_CLIP), r, atol=1e-5)),
    ('gana la clase de mayor similitud coseno', lambda r: (r.argmax(1) == (F.normalize(_ii, dim=-1) @ F.normalize(_ti, dim=-1).T).argmax(1)).all()),
    ('coincide con la referencia (y usa la escala)', lambda r: torch.allclose(r, _REF['zero_shot'](_ii, _ti, ESCALA_CLIP), atol=1e-5)
                                                       and not torch.allclose(zero_shot(_ii, _ti, 1.0), r, atol=1e-3)),
])

In [ ]:
# 5.2 Zero-shot en tres datasets y tres maneras de escribir las clases
PLANTILLAS = ['a photo of a {}.', 'a blurry photo of a {}.', 'a photo of the large {}.', 'a photo of the small {}.',
              'a low resolution photo of a {}.', 'a black and white photo of a {}.', 'a close-up photo of a {}.', 'a drawing of a {}.']
def clasificador_texto(clases, plantillas):
    """Un embedding por clase: el promedio de los embeddings (normalizados) de todas las plantillas."""
    filas = [F.normalize(emb_textos([p.format(c) for p in plantillas]), dim=-1).mean(0) for c in clases]
    return torch.stack(filas)

zs = usar('E10', 'zero_shot')
ESTRATEGIAS = {'solo el nombre': ['{}'], '"a photo of a {}."': ['a photo of a {}.'], f'{len(PLANTILLAS)} plantillas promediadas': PLANTILLAS}
RES_ZS = {}
for conj, (ims, ys, clases) in CONJUNTOS.items():
    for est, pl in ESTRATEGIAS.items():
        RES_ZS[conj, est] = (zs(EMB[conj], clasificador_texto(clases, pl), ESCALA_CLIP).argmax(1).numpy() == ys).mean()
REF_T2 = {'CIFAR-10': 0.77, 'Imagenette': 0.92, 'Fashion-MNIST': 0.79}   # Tarea 2: ResNet-18 congelada + 500 etiquetas

print(f'{"":15s}' + ''.join(f'{e:>28s}' for e in ESTRATEGIAS) + f'{"ResNet-18 + 500 etiquetas":>28s}')
for conj in CONJUNTOS:
    print(f'{conj:15s}' + ''.join(f'{RES_ZS[conj, e]:28.1%}' for e in ESTRATEGIAS) + f'{"≈ " + format(REF_T2[conj], ".0%"):>28s}')

x = np.arange(len(CONJUNTOS)); w = 0.2
plt.figure(figsize=(9, 3.6))
for k, e in enumerate(ESTRATEGIAS):
    plt.bar(x + (k - 1) * w, [RES_ZS[c, e] for c in CONJUNTOS], w, label=f'CLIP zero-shot · {e}')
plt.scatter(x, [REF_T2[c] for c in CONJUNTOS], marker='_', s=900, color='k', label='Tarea 2: ResNet-18 + 500 etiquetas', zorder=3)
plt.xticks(x, list(CONJUNTOS)); plt.ylim(0, 1); plt.ylabel('exactitud'); plt.legend(fontsize=7, loc='upper left', bbox_to_anchor=(1.01, 1)); plt.tight_layout(); plt.show()

**Para conversar.**
- En CIFAR-10 e Imagenette, CLIP **sin ver una sola etiqueta** queda a la altura de (o supera a) la ResNet-18 de la Tarea 2, que necesitó 500 imágenes etiquetadas y entrenar una cabeza.
- La forma de escribir la clase importa. En fotos, "a photo of a ..." suele ayudar y promediar varias plantillas ayuda más todavía; en Fashion-MNIST, que no son fotos, "a photo of a ..." puede incluso empeorar. A esto se le llama *prompt engineering*, y CLIP lo heredó de cómo se escribe el texto en internet.
- **Fashion-MNIST es donde CLIP rinde peor y donde menos ventaja le saca a la Tarea 2.** Siluetas de 28×28 en gris, sobre fondo negro, casi no existen en los datos con que se entrenó CLIP. El zero-shot no hace magia fuera de su distribución, la misma lección que dejó la Tarea 2 con el *transfer learning*.

In [ ]:
# 5.3 ¿Cuántas fotos etiquetadas "vale" una frase? Sonda lineal con k fotos por clase vs zero-shot
from sklearn.linear_model import LogisticRegression
tr_im, tr_y, _ = leer_carpeta(ZIPS['cifar10'], 'cifar10/train', 48, semilla=1)
EMB_TR = F.normalize(emb_imagenes(tr_im), dim=-1).numpy()
EMB_TE = F.normalize(EMB['CIFAR-10'], dim=-1).numpy()
KS, curva = [1, 2, 4, 8, 16], []
for k in KS:
    accs = []
    for sem in range(3):
        rng = np.random.default_rng(sem)
        sel = np.concatenate([rng.choice(np.nonzero(tr_y == c)[0], k, replace=False) for c in range(10)])
        sonda = LogisticRegression(C=10, max_iter=3000).fit(EMB_TR[sel], tr_y[sel])
        accs.append((sonda.predict(EMB_TE) == cifar_y).mean())
    curva.append(np.mean(accs))
    print(f'{k:2d} fotos por clase: {curva[-1]:.1%}')
plt.figure(figsize=(5.5, 3.5))
plt.plot(KS, curva, 'o-', label='sonda lineal sobre CLIP (k fotos por clase)')
plt.axhline(RES_ZS['CIFAR-10', f'{len(PLANTILLAS)} plantillas promediadas'], color='C3', ls='--', label='zero-shot (0 fotos)')
plt.xscale('log', base=2); plt.xticks(KS, KS); plt.xlabel('fotos etiquetadas por clase'); plt.ylabel('exactitud en CIFAR-10')
plt.legend(fontsize=8); plt.tight_layout(); plt.show()

En el paper de CLIP, promediando sobre 27 datasets, el zero-shot rinde más o menos lo mismo que una sonda lineal entrenada con **4 fotos por clase** sobre las mismas *features*. Mira en qué punto de la curva cruza aquí la línea roja. Una frase bien escrita vale varias fotos etiquetadas; con suficientes fotos, la sonda termina ganando.

In [ ]:
# 5.4 Buscar imágenes con frases en COCO128
nombres_coco = sorted(n.split('/')[-1][:-4] for n in ZIPS['coco128'].namelist() if n.endswith('.jpg'))
fotos_coco = [foto(n) for n in nombres_coco]
EMB_COCO = F.normalize(emb_imagenes(fotos_coco), dim=-1)
BUSQUEDAS = ['a giraffe', 'a cat lying down', 'people playing tennis', 'an old black and white photograph', 'food on a plate', 'something to sit on']
sims = EMB_COCO @ F.normalize(emb_textos(BUSQUEDAS), dim=-1).T
fig, ax = plt.subplots(len(BUSQUEDAS), 3, figsize=(8, 2.3 * len(BUSQUEDAS)))
for i, q in enumerate(BUSQUEDAS):
    for j, k in enumerate(sims[:, i].argsort(descending=True)[:3].tolist()):
        ax[i, j].imshow(a_cuadrado(fotos_coco[k], 200)); ax[i, j].axis('off'); ax[i, j].set_title(f'{sims[k, i]:.3f}', fontsize=8)
    ax[i, 0].text(-20, 100, f'"{q}"', ha='right', va='center', fontsize=10)
plt.tight_layout(); plt.show()

Nadie etiquetó "algo para sentarse" ni "fotografía antigua en blanco y negro": CLIP lo resuelve porque aprendió de texto libre. Esto es la base de los buscadores de imágenes modernos (y de la detección y segmentación de "vocabulario abierto" que veremos en la sección 7).

### El precio de haber aprendido de internet: el ataque tipográfico
CLIP aprendió que el texto escrito en una imagen suele describirla. Eso lo hace sensible a una trampa absurda.

In [ ]:
# 5.5 Ataque tipográfico: escribir "DOG" sobre un gato
def escribir(im, texto):
    im = im.copy(); d = ImageDraw.Draw(im); W, H = im.size
    fuente = ImageFont.truetype(font_manager.findfont(font_manager.FontProperties(family='DejaVu Sans', weight='bold')), size=H // 5)
    x0, y0, x1, y1 = d.textbbox((0, 0), texto, font=fuente)
    px, py = (W - (x1 - x0)) // 2, int(H * 0.35)                    # un cartel blanco sobre el cuerpo del gato
    d.rectangle([px - 12, py - 8, px + (x1 - x0) + 12, py + (y1 - y0) + 18], fill='white')
    d.text((px - x0, py - y0), texto, fill='black', font=fuente)
    return im

gato = foto(FOTO_GATO)
VERSIONES = {'original': gato, '"DOG"': escribir(gato, 'DOG'), '"PERRO"': escribir(gato, 'PERRO')}
OPCIONES = ['a photo of a cat', 'a photo of a dog', 'a photo of a car']
P_ataque = zs(emb_imagenes(list(VERSIONES.values())), emb_textos(OPCIONES), ESCALA_CLIP)
fig, ax = plt.subplots(2, 3, figsize=(12, 6), gridspec_kw=dict(height_ratios=[3, 1.2]))
for j, (nombre, im) in enumerate(VERSIONES.items()):
    ax[0, j].imshow(im); ax[0, j].axis('off'); ax[0, j].set_title(nombre)
    ax[1, j].barh(OPCIONES, P_ataque[j].numpy(), color=['C0', 'C3', 'C7']); ax[1, j].set_xlim(0, 1)
plt.tight_layout(); plt.show()
for nombre, p in zip(VERSIONES, P_ataque):
    print(f'{nombre:10s}', '  '.join(f'{o}: {v:.1%}' for o, v in zip(OPCIONES, p.tolist())))

Un cartel con la palabra "DOG" puede pesar más que el gato que se ve detrás. Fíjate también en "PERRO": CLIP aprendió sobre todo de texto en inglés, así que la palabra en castellano le dice mucho menos. El modelo no "lee" en el sentido humano; asocia formas de letras con conceptos porque en internet suelen aparecer juntas.

In [ ]:
# 5.6 CLIP como juez de Stable Diffusion: ¿la guía hace que la imagen "diga" más lo que pide el texto?
if 'IMAGENES_GUIA' in globals():
    e_im = F.normalize(emb_imagenes(list(IMAGENES_GUIA.values())), dim=-1)
    e_tx = F.normalize(emb_textos([PROMPT_GUIA]), dim=-1)
    clip_score = (100 * e_im @ e_tx.T).squeeze(1).clamp(min=0)          # CLIPScore (Hessel et al., 2021)
    for s, c in zip(IMAGENES_GUIA, clip_score):
        print(f'escala {s:>4g}: CLIPScore = {c:.1f}')
    plt.figure(figsize=(5, 3)); plt.plot(list(IMAGENES_GUIA), clip_score.numpy(), 'o-')
    plt.xlabel('escala de guía'); plt.ylabel('CLIPScore'); plt.title(PROMPT_GUIA, fontsize=8); plt.tight_layout(); plt.show()
else:
    print('⏭ Esta celda necesita las imágenes de la sección 4.5.')

Así se evalúan los generadores de texto a imagen en la práctica: **FID** para el realismo (sección 3) y **CLIPScore** para la fidelidad al texto. Lo normal es que el CLIPScore suba con la guía y después se estanque, mientras que el realismo empeora con escalas altas.

## 6 · Conversar con imágenes: BLIP-2 y LLaVA (15 min)

CLIP compara imágenes con frases, pero no **escribe**. Para que un modelo de lenguaje (LLM) "vea", hay que traducir la imagen a algo que el LLM entienda: vectores del mismo tamaño que sus *embeddings* de palabras. Dos diseños influyentes:

- **BLIP-2** (Li et al., 2023): encoder visual **congelado** + LLM **congelado**, unidos por un **Q-Former**: 32 *queries* aprendibles que, con atención cruzada, resumen la imagen en **32 vectores**. Solo se entrena el puente.
- **LLaVA** (Liu et al., 2023): mucho más simple. Cada parche del encoder visual pasa por una proyección (una capa lineal en el LLaVA original; un **MLP de dos capas** desde LLaVA-1.5) y entra al LLM como si fuera una palabra más. Si el encoder entrega 729 parches, la imagen ocupa **729 tokens**.

In [ ]:
# 6.1 La anatomía de BLIP-2 sin descargar sus 15 GB: solo el config.json y una red "fantasma" en el dispositivo meta
from transformers import Blip2Config, Blip2ForConditionalGeneration
cfg_b2 = Blip2Config.from_pretrained('Salesforce/blip2-opt-2.7b')
with torch.device('meta'):                         # crea la arquitectura sin reservar memoria para los pesos
    blip2 = Blip2ForConditionalGeneration(cfg_b2)
def millones(m):
    return sum(p.numel() for p in m.parameters()) / 1e6
PARTES_B2 = {'Encoder visual EVA ViT-g (congelado)': millones(blip2.vision_model),
             'Q-Former + 32 queries (se entrena)': millones(blip2.qformer) + blip2.query_tokens.numel() / 1e6,
             'Proyección al LLM (se entrena)': millones(blip2.language_projection),
             'LLM OPT-2.7B (congelado)': millones(blip2.language_model)}
total_b2 = sum(PARTES_B2.values())
for k, v in PARTES_B2.items():
    print(f'{k:40s} {v:8.1f} M  ({v / total_b2:5.1%})')
print(f'La imagen entra al LLM como {cfg_b2.num_query_tokens} vectores, cualquiera sea su contenido.')
del blip2

In [ ]:
# 6.2 LLaVA pequeño: SigLIP (encoder visual) + Qwen 0,5B (LLM), ≈0,9 B de parámetros en total
from transformers import AutoProcessor, LlavaForConditionalGeneration
LLAVA_ID = 'llava-hf/llava-interleave-qwen-0.5b-hf'
if not GPU and 'pipe' in globals():
    globals().pop('pipe'); gc.collect()            # en CPU liberamos Stable Diffusion para que alcance la RAM
    print('(CPU) Stable Diffusion liberado de la memoria.')
llava = LlavaForConditionalGeneration.from_pretrained(LLAVA_ID, torch_dtype=torch.float16 if GPU else torch.float32).to(dispositivo).eval()
proc_llava = AutoProcessor.from_pretrained(LLAVA_ID)
ID_IMAGEN = getattr(llava.config, 'image_token_id', None) or llava.config.image_token_index
proyector = dict(llava.named_modules())[next(n for n, _ in llava.named_modules() if n.endswith('multi_modal_projector'))]
CAPTURA = {}
proyector.register_forward_hook(lambda m, ent, sal: CAPTURA.update(entrada=ent[0].detach().cpu(), salida=sal.detach().cpu()))

@torch.no_grad()
def preguntar(imagen, pregunta, max_tokens=60):
    if next(llava.parameters()).device.type != dispositivo.type:
        llava.to(dispositivo)                      # por si quedó en la RAM (ver el final de 6.3)
    conv = [{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': pregunta}]}]
    texto = proc_llava.apply_chat_template(conv, add_generation_prompt=True)
    ent = proc_llava(images=imagen, text=texto, return_tensors='pt').to(dispositivo)
    ent['pixel_values'] = ent['pixel_values'].to(llava.dtype)
    CAPTURA['ids'] = ent['input_ids']
    sal = llava.generate(**ent, max_new_tokens=max_tokens, do_sample=False)
    return proc_llava.decode(sal[0, ent['input_ids'].shape[1]:], skip_special_tokens=True).strip()

grupo = foto(FOTO_GRUPO)
print('LLaVA:', preguntar(grupo, 'Describe this image in one sentence.'))
n_img = int((CAPTURA['ids'] == ID_IMAGEN).sum()); n_tot = CAPTURA['ids'].shape[1]
print(f'\nEl prompt tiene {n_tot} tokens: {n_img} son de la imagen ({n_img / n_tot:.0%}) y {n_tot - n_img} de texto.')
print('Encoder visual -> proyector -> LLM:', tuple(CAPTURA['entrada'].shape), '->', tuple(CAPTURA['salida'].shape))

In [ ]:
# 🧩 E11 · El puente de LLaVA, a mano
def proyectar(feats, proyector):
    h = feats @ proyector.linear_1.weight.T + proyector.linear_1.bias
    h = F.gelu(h)
    return h @ proyector.linear_2.weight.T + proyector.linear_2.bias

In [ ]:
# ✔️ Verificación E11 (contra la salida real del proyector, capturada con un hook en 6.2)
_llamadas = []
_h = [proyector.register_forward_hook(lambda *a: _llamadas.append(1)),
      proyector.linear_1.register_forward_hook(lambda *a: _llamadas.append(1))]
try:
    _r11 = proyectar(CAPTURA['entrada'].to(proyector.linear_1.weight.device), proyector)
finally:
    for h in _h: h.remove()
revisar('E11', lambda: _r11, [
    ('forma (1, N, D_llm)', lambda r: tuple(r.shape) == tuple(CAPTURA['salida'].shape)),
    ('coincide con la salida real del proyector', lambda r: torch.allclose(r.float().cpu(), CAPTURA['salida'].float(), atol=1e-2 if GPU else 1e-4, rtol=1e-2 if GPU else 1e-4)),
    ('no llama a los módulos (solo usa los pesos)', lambda r: len(_llamadas) == 0),
])
if funciona('E11'):
    print(f'El puente entero tiene {millones(proyector):.1f} M de parámetros, contra {PARTES_B2["Q-Former + 32 queries (se entrena)"]:.0f} M del Q-Former de BLIP-2.')

In [ ]:
# 6.3 Preguntas difíciles
gato_dog = VERSIONES['"DOG"']
PREGUNTAS = [
    (grupo, 'How many people are in this image? Answer with a number.', 'Hay 12 personas (tutorial 09).'),
    (grupo, "What color is the dog's collar?", 'No hay ningún perro en la foto.'),
    (gato_dog, 'What animal is in this picture, and what is written on it?', 'El ataque tipográfico de 5.5.'),
]
fig, ax = plt.subplots(1, len(PREGUNTAS), figsize=(5 * len(PREGUNTAS), 4))
for a, (im, q, nota) in zip(ax, PREGUNTAS):
    r = preguntar(im, q, max_tokens=40)
    a.imshow(im); a.axis('off'); a.set_title(f'P: {q}\nR: {r}\n({nota})', fontsize=8, loc='left')
    print(f'P: {q}\nR: {r}\n')
plt.tight_layout(); plt.show()
if GPU:
    llava.to('cpu')                                # LLaVA espera en la RAM; preguntar() lo devuelve a la GPU en el cierre
memoria_gpu('con LLaVA en la RAM')

**Para conversar.** Es un modelo pequeño (0,5 B en el LLM), así que sus límites se notan, pero los grandes fallan en lo mismo con menos frecuencia:
- **Contar** sigue siendo difícil: los parches se vuelven tokens, pero nada en la arquitectura obliga a separar instancias (lo que sí hacía la detección del tutorial 09).
- **Alucinar:** si la pregunta presupone algo que no está ("el collar del perro"), el modelo tiende a inventar una respuesta en vez de corregir la premisa.
- A diferencia de CLIP, un LLM con visión puede **leer** el texto de la imagen y, a la vez, reconocer al animal. Leer bien no impide equivocarse por otras razones.

## 7 · SAM + CLIP: segmentar lo que nombras (15 min)

**SAM** (*Segment Anything*, Kirillov et al., 2023) es un modelo fundacional de segmentación. Es *promptable*: recibe un **punto**, una **caja** o una máscara aproximada y devuelve la máscara del objeto. Su diseño separa el costo: un encoder ViT pesado que se corre **una vez por imagen** y un decodificador liviano que responde a cada prompt en milisegundos. Se entrenó con SA-1B: 1.100 millones de máscaras en 11 millones de imágenes.

SAM sabe **dónde** están las cosas, pero no **qué** son. CLIP sabe **qué** es algo, pero no **dónde** está. Juntos dan segmentación de **vocabulario abierto**.

Usamos la implementación de Ultralytics (la de YOLO11 del tutorial 09), que descarga los pesos desde GitHub: `sam_b.pt` en GPU y `mobile_sam.pt` (un encoder destilado, mucho más liviano) en CPU.

In [ ]:
# 7.1 SAM con un punto, con un punto + un punto negativo, y con una caja
from ultralytics import SAM
sam = SAM(CFG['sam'])
g_np = np.array(grupo)
PUNTO, PUNTO_NO, CAJA = [257, 283], [173, 292], [222, 190, 292, 376]

def mascara_de(resultado):
    return resultado.masks.data[0].cpu().numpy().astype(bool)

def superponer(img, mascaras, alfa=0.55, semilla=0, color=None):
    """Pinta cada máscara con un color al azar (o con `color`, si se da)."""
    rng = np.random.default_rng(semilla); salida = img.astype(np.float32).copy()
    for m in mascaras:
        color_m = np.array(color, np.float32) if color is not None else rng.uniform(40, 255, 3)
        salida[m] = (1 - alfa) * salida[m] + alfa * color_m
    return salida.astype(np.uint8)

t0 = time.time()
m_punto = mascara_de(sam(g_np, points=[PUNTO], labels=[1], verbose=False)[0])
m_dos = mascara_de(sam(g_np, points=[[PUNTO, PUNTO_NO]], labels=[[1, 0]], verbose=False)[0])
m_caja = mascara_de(sam(g_np, bboxes=[CAJA], verbose=False)[0])
print(f'3 consultas en {time.time() - t0:.1f} s (Ultralytics vuelve a codificar la imagen en cada llamada)')
fig, ax = plt.subplots(1, 3, figsize=(15, 4.4))
for a, m, titulo in zip(ax, [m_punto, m_dos, m_caja], ['un punto (verde)', 'punto positivo + negativo (rojo)', 'una caja']):
    a.imshow(superponer(g_np, [m])); a.axis('off'); a.set_title(titulo)
ax[0].plot(*PUNTO, 'g*', ms=16); ax[1].plot(*PUNTO, 'g*', ms=16); ax[1].plot(*PUNTO_NO, 'r*', ms=16)
ax[2].add_patch(plt.Rectangle(CAJA[:2], CAJA[2] - CAJA[0], CAJA[3] - CAJA[1], fill=False, color='lime', lw=2))
plt.tight_layout(); plt.show()

In [ ]:
# 7.2 "Segmentar todo": una grilla de puntos como prompts, y luego quitar duplicados
def mascaras_automaticas(img, lado=CFG['sam_grilla'], iou_max=0.7, area_min=0.002):
    """Un prompt por punto de una grilla lado×lado. Devuelve (máscaras (M,H,W) bool, puntajes (M,))."""
    H, W = img.shape[:2]
    puntos = [[(j + 0.5) * W / lado, (i + 0.5) * H / lado] for i in range(lado) for j in range(lado)]
    bloques_M, bloques_s = [], []
    for i in range(0, len(puntos), 64):            # de a 64 prompts: con 256 juntos la GPU se queda sin memoria
        lote = puntos[i:i + 64]
        r = sam(img, points=lote, labels=[1] * len(lote), verbose=False)[0]
        if r.masks is None:
            continue
        bloques_M.append(r.masks.data.cpu().numpy().astype(bool))
        bloques_s.append(r.boxes.conf.cpu().numpy() if r.boxes is not None else np.ones(len(bloques_M[-1])))
        del r
    if GPU:
        torch.cuda.empty_cache()
    if not bloques_M:
        return np.zeros((0, H, W), bool), np.zeros(0)
    M, s = np.concatenate(bloques_M), np.concatenate(bloques_s)
    chicas = M[:, ::4, ::4].reshape(len(M), -1)              # IoU sobre una versión reducida: mucho más rápido
    areas = chicas.sum(1); elegidas = []
    for i in np.argsort(-s):                                  # NMS con IoU de máscaras, como el del tutorial 09
        if M[i].sum() < area_min * H * W:
            continue
        if all((chicas[i] & chicas[j]).sum() / max(1, (chicas[i] | chicas[j]).sum()) < iou_max for j in elegidas):
            elegidas.append(i)
    return M[elegidas], s[elegidas]

t0 = time.time()
M_grupo, S_grupo = mascaras_automaticas(g_np)
print(f'{CFG["sam_grilla"] ** 2} puntos -> {len(M_grupo)} máscaras distintas en {time.time() - t0:.1f} s')
plt.figure(figsize=(8, 6)); plt.imshow(superponer(g_np, M_grupo)); plt.axis('off'); plt.title('SAM sin saber qué es cada cosa'); plt.show()

In [ ]:
# 🧩 E12 · Recortar una máscara para mostrársela a CLIP
def recortar_mascara(imagen, mascara, margen=0.1, aislar=True):
    ys, xs = np.nonzero(mascara)
    y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
    my, mx = int(round(margen * (y1 - y0))), int(round(margen * (x1 - x0)))
    H, W = mascara.shape
    y0, y1, x0, x1 = max(0, y0 - my), min(H, y1 + my), max(0, x0 - mx), min(W, x1 + mx)
    rec = imagen[y0:y1, x0:x1].copy()
    if aislar:
        rec[~mascara[y0:y1, x0:x1]] = 127
    return rec

In [ ]:
# ✔️ Verificación E12
_img = np.random.default_rng(0).integers(0, 256, (100, 200, 3), dtype=np.uint8); _img_copia = _img.copy()
_m = np.zeros((100, 200), bool); _m[20:40, 50:90] = True                       # caja de 20×40
_yy, _xx = np.mgrid[:100, :200]; _circ = (_yy - 50) ** 2 + (_xx - 100) ** 2 < 15 ** 2
_borde = np.zeros((100, 200), bool); _borde[0:10, 190:200] = True
revisar('E12', lambda: recortar_mascara(_img, _m, 0.1, True), [
    ('caja de 20×40 con margen 0,1 -> recorte de 24×48×3', lambda r: r.shape == (24, 48, 3) and r.dtype == np.uint8),
    ('aislar=True: fuera de la máscara queda gris 127', lambda r: (r[0, 0] == 127).all() and (r[-1, -1] == 127).all()),
    ('dentro de la máscara se conservan los píxeles', lambda r: (r[2, 4] == _img[20, 50]).all() and (r[21, 43] == _img[39, 89]).all()),
    ('aislar=False: es el recorte tal cual', lambda r: np.array_equal(recortar_mascara(_img, _m, 0.1, False), _img[18:42, 46:94])),
    ('una máscara circular deja gris las esquinas de su caja', lambda r: (recortar_mascara(_img, _circ, 0.0, True)[0, 0] == 127).all()),
    ('no se sale de la imagen en los bordes', lambda r: recortar_mascara(_img, _borde, 0.5, False).shape == (15, 15, 3)),
    ('no modifica la imagen original', lambda r: np.array_equal(_img, _img_copia)),
])

In [ ]:
# 7.3 Segmentar por texto: SAM propone, CLIP elige
OTRAS = ['the ground', 'the sky', 'a wall', 'a building', 'a tree', 'grass', 'a blurry background']
def segmentar_por_texto(img, mascaras, consultas, aislar=True, umbral=0.5):
    """Para cada consulta, las máscaras cuyo recorte CLIP asigna a esa consulta con probabilidad > umbral."""
    rec = usar('E12', 'recortar_mascara')
    recortes = [Image.fromarray(rec(img, m, 0.1, aislar)) for m in mascaras]
    textos = [f'a photo of {c}' for c in consultas] + [f'a photo of {o}' for o in OTRAS]
    P = usar('E10', 'zero_shot')(emb_imagenes(recortes), emb_textos(textos), ESCALA_CLIP)
    gana = P.argmax(1).numpy()
    return {c: [i for i in range(len(mascaras)) if gana[i] == k and P[i, k] > umbral] for k, c in enumerate(consultas)}

CONSULTAS = ['a person', 'an airplane']
COLORES = [(255, 0, 200), (0, 220, 255)]
res_aislar = segmentar_por_texto(g_np, M_grupo, CONSULTAS, aislar=True)
res_contexto = segmentar_por_texto(g_np, M_grupo, CONSULTAS, aislar=False)
fig, ax = plt.subplots(1, 4, figsize=(20, 4.4))
for j, (res, modo) in enumerate([(res_aislar, 'recorte aislado'), (res_contexto, 'recorte con fondo')]):
    for k, c in enumerate(CONSULTAS):
        a = ax[2 * j + k]; a.imshow(superponer(g_np, [M_grupo[i] for i in res[c]], color=COLORES[k])); a.axis('off')
        a.set_title(f'"{c}" · {modo}: {len(res[c])} máscaras', fontsize=10)
plt.tight_layout(); plt.show()
print('Personas según las etiquetas de COCO: 12. Máscaras "a person": aislado =', len(res_aislar['a person']), '· con fondo =', len(res_contexto['a person']))

jir_np = np.array(foto(FOTO_JIRAFA))
M_jir, _ = mascaras_automaticas(jir_np)
res_jir = segmentar_por_texto(jir_np, M_jir, ['a giraffe', 'a tree trunk'])
mostrar([superponer(jir_np, [M_jir[i] for i in res_jir[c]], color=COLORES[k]) for k, c in enumerate(res_jir)], [f'"{c}"' for c in res_jir], tam=5)

**Para conversar.** Funciona sin haber entrenado nada, pero mira los detalles:
- SAM entrega **partes y todos** a la vez (un abrigo, una persona, un grupo), así que "contar máscaras" no es "contar personas". El umbral y el NMS deciden mucho.
- **Recortar con o sin fondo** cambia lo que CLIP ve: el fondo da contexto, pero también arrastra a la clase de lo que está alrededor.
- **La frontera.** SAM 2 (2024) extiende SAM a **video** con una memoria que sigue al objeto entre cuadros. SAM 3 (2025) acepta directamente **conceptos** ("todos los gatos atigrados") y hace en un solo modelo lo que acabamos de armar a mano. Y para que un agente **actúe** en el mundo físico no basta con nombrar y segmentar: necesita predecir cómo cambia la escena. Ese es el programa de los *world models*, como V-JEPA 2 (Meta, 2025), que predice **representaciones** del futuro (no píxeles) con la misma idea de profesor EMA de DINO (tutorial 13).

## 8 · Cierre: el circuito completo (5 min)

Una frase se convierte en imagen (Stable Diffusion), CLIP mide cuánto se parece al texto, LLaVA la describe con sus palabras, y SAM + CLIP segmentan el objeto que nombramos.

In [ ]:
# 8.1 Texto -> imagen -> texto -> máscara
PROMPT_FINAL, OBJETO = 'a photo of a red bicycle leaning against a stone wall', 'a bicycle'
if 'pipe' in globals():
    pipe.to(dispositivo)                           # SD vuelve a la GPU solo para esta imagen
    g = torch.Generator(device=dispositivo).manual_seed(0)
    img_final = pipe(PROMPT_FINAL, num_inference_steps=CFG['sd_pasos'], guidance_scale=7.5, generator=g, height=LADO, width=LADO).images[0]
    pipe.to('cpu'); memoria_gpu('con SD de vuelta en la RAM')
else:                                                    # en CPU liberamos SD en la sección 6: reutilizamos una imagen de 4.5
    PROMPT_FINAL, OBJETO, img_final = PROMPT_GUIA, 'a boot', IMAGENES_GUIA[7.5]
score = (100 * F.normalize(emb_imagenes([img_final]), dim=-1) @ F.normalize(emb_textos([PROMPT_FINAL]), dim=-1).T).item()
descripcion = preguntar(img_final, 'Describe this image in one sentence.')
f_np = np.array(img_final)
M_f, _ = mascaras_automaticas(f_np)
sel = segmentar_por_texto(f_np, M_f, [OBJETO])[OBJETO]
fig, ax = plt.subplots(1, 2, figsize=(11, 5.2))
ax[0].imshow(img_final); ax[0].axis('off'); ax[0].set_title(f'Prompt: {PROMPT_FINAL}\nCLIPScore = {score:.1f}', fontsize=9)
ax[1].imshow(superponer(f_np, [M_f[i] for i in sel], color=COLORES[0])); ax[1].axis('off')
ax[1].set_title(f'LLaVA: {descripcion[:90]}\nSAM + CLIP: "{OBJETO}" -> {len(sel)} máscaras', fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# 8.2 Estado de tus ejercicios
NOMBRES = {'E1': 'reparametrizar', 'E2': 'perdida_vae', 'E3': 'perdidas_gan', 'E4': 'difundir', 'E5': 'paso_ddpm', 'E6': 'guiar',
           'E7': 'distancia_frechet', 'E8': 'generar_a_mano', 'E9': 'perdida_clip', 'E10': 'zero_shot', 'E11': 'proyectar', 'E12': 'recortar_mascara'}
for e, n in NOMBRES.items():
    print(f'{ESTADO.get(e, "⬜")}  {e:4s} {n}')
print(f'\n{sum(funciona(e) for e in NOMBRES)}/{len(NOMBRES)} ejercicios verificados.')

### Lo que te llevas
| Idea | Dónde la viste hoy | Dónde la vas a encontrar |
|---|---|---|
| Un espacio latente ordenado sirve para generar | VAE (1) | el VAE de Stable Diffusion (4) |
| Aprender a quitar ruido = aprender a generar | difusión desde cero (3) | SD, FLUX, Sora |
| La guía cambia fidelidad por diversidad | botines (3.5) y `guidance_scale` (4.5) | todo generador condicionado |
| FID y CLIPScore miden cosas distintas | 3.6 y 5.6 | papers de generación |
| Alinear dos modalidades con aprendizaje contrastivo | CLIP (5) | búsqueda, zero-shot, el texto de SD |
| Una imagen se vuelve tokens para un LLM | Q-Former vs proyector (6) | asistentes multimodales |
| Combinar modelos fundacionales sin entrenar | SAM + CLIP (7) | SAM 3, detección de vocabulario abierto |

### ✅ Checklist
1. ¿Por qué decodificar un punto al azar del autoencoder suele dar basura y en el VAE no? ¿Qué término de la pérdida es responsable?
2. ¿Qué problema resuelve el truco no saturante de la GAN? ¿Por qué es peor justo al comienzo del entrenamiento?
3. Explica con tus palabras por qué `difundir` permite entrenar la red de difusión sin recorrer los 1000 pasos.
4. ¿Qué hace la guía libre de clasificador y qué se pierde al subir la escala?
5. FID bajo y CLIPScore alto: ¿qué mide cada uno? Inventa un generador que tenga uno bueno y el otro malo.
6. ¿Por qué Stable Diffusion hace la difusión en el latente del VAE y no en los píxeles?
7. CLIP clasifica sin etiquetas pero falla en Fashion-MNIST y cae en el ataque tipográfico. ¿Qué tienen en común ambos fallos?

### 🚀 Propuestos (fuera de las 3 horas)
1. **β-VAE.** Entrena el VAE con `beta` = 0,25 y 4 (E2 ya lo acepta). ¿Cómo cambian la nitidez y el orden del espacio latente?
2. **Horario coseno** (Nichol y Dhariwal, 2021) en vez del lineal. Compara el FID-lite con DDIM de 10, 25 y 50 pasos.
3. **Prompt negativo.** Modifica `generar_a_mano` para que, en vez de `""`, la predicción "sin condición" use un texto como `"blurry, low quality"`. ¿Qué cambia?
4. **CLIP más grande.** Repite 5.2 con `openai/clip-vit-large-patch14`. ¿Mejora Fashion-MNIST? ¿Y el ataque tipográfico?
5. **SAM guiado por un detector.** Usa las cajas de YOLO11 (tutorial 09) como prompts de SAM y compara el conteo de personas con el de 7.3.

### Referencias
- Kingma y Welling (2014). *Auto-Encoding Variational Bayes*.
- Goodfellow et al. (2014). *Generative Adversarial Nets*. · Radford, Metz y Chintala (2016). *DCGAN*.
- Ho, Jain y Abbeel (2020). *Denoising Diffusion Probabilistic Models*. · Song, Meng y Ermon (2021). *DDIM*.
- Ho y Salimans (2022). *Classifier-Free Diffusion Guidance*. · Heusel et al. (2017). *GANs Trained by a Two Time-Scale Update Rule* (FID).
- Rombach et al. (2022). *High-Resolution Image Synthesis with Latent Diffusion Models*. · Meng et al. (2022). *SDEdit*.
- Radford et al. (2021). *Learning Transferable Visual Models From Natural Language Supervision* (CLIP). · Hessel et al. (2021). *CLIPScore*.
- Li et al. (2023). *BLIP-2*. · Liu et al. (2023). *Visual Instruction Tuning* (LLaVA).
- Kirillov et al. (2023). *Segment Anything*. · Ravi et al. (2024). *SAM 2*. · Carion et al. (2025). *SAM 3*.
- Assran et al. (2025). *V-JEPA 2*.